# CA Experiment 6 — From-Scratch ~80M ADA Daily-QA Agent (Persona-Bearing, QPM-in-scope)

**Runtime:** Colab Pro **A100 GPU** + bounded Anthropic API (Sonnet persona/style/refusal data + Sonnet 4.5 judge) + free open corpora.

One from-scratch ~80M decoder-only model as ADA's **daily-QA** agent: reads retrieved context → answers/abstains **in character**. Persona/SCI/SMC are trained, and the **QPM output is compiled into the weights** via a `persona_state` `<|persona|>` channel (plan §5.5).

| Axis | In scope | Measured by |
|---|---|---|
| Grounded-QA competence (H1) | ✅ | correct-and-grounded % + SQuAD2 EM/F1 |
| Calibrated abstention / SMC-C (H2) | ✅ | abstention F1 + hallucination rate |
| Persona T/E/C/S (H3) | ✅ | Exp 1–5 PersonaScore harness (Sonnet judge) |
| SCI-refresh policy (H4) | ✅ | R0 vs R1 (refresh @15/30) |
| Fallback gap (H5/RQ5) | ✅ | vs fine-tuned SmolLM2-135M |
| **QPM-as-weight-supervision (RQ6)** | ✅ | QPM-on vs `--no-qpm` ablation |

The notebook is **resumable**: checkpoints mirror to Drive; re-running a training cell resumes with `--resume`. Timeline: **P0** provision → **P1** tokenizer + pilot gate → **P2** Stage-A pretrain → **P3** Stage-B SFT → **P4** baseline → **P5** eval → **P6** decision.

## Cell 1: Mount Drive, set paths, load API key

Exp 6 is **self-contained** (no cross-experiment assets). The Anthropic key is only needed for the Sonnet persona-data and judging cells — all training/corpus steps are free.

In [46]:
# from google.colab import drive, userdata
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_6'
CKPT_DIR    = f'{PROJECT_DIR}/checkpoints'   # mirrored to Drive (survives disconnects)
DATA_DIR    = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'

# Pretrain bins live on LOCAL Colab SSD for training (random-access memmap reads over the
# Drive FUSE mount are far too slow and stall Stage-A). A one-time sequential copy to the
# Drive archive below keeps a durable, downloadable copy — see cell P1b.
LOCAL_PT    = '/content/data/pretrain'       # fast local disk (ephemeral)
DRIVE_PT    = f'{DATA_DIR}/pretrain'         # durable archive on Drive (persistent, downloadable)

assert os.path.exists(PROJECT_DIR), f'Upload CA_Experiment_6 to Drive first! Not found at {PROJECT_DIR}'
for d in (CKPT_DIR, DRIVE_PT, f'{DATA_DIR}/persona_scripts', RESULTS_DIR, LOCAL_PT):
    os.makedirs(d, exist_ok=True)


def _parse_env(path):
    """Tolerant .env parser: handles BOM, blank/comment lines, `export `, quotes, spaces."""
    vals = {}
    for raw in open(path, encoding='utf-8-sig'):
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        k, v = line.split('=', 1)
        vals[k.strip()] = v.strip().strip('"').strip("'")
    return vals


# API key — Colab Secrets preferred, then a .env on Drive (same pattern as Exp 4/5).
ANTHROPIC_API_KEY = ''
for secret_name in ('CHA_EXPERIMENT_SONNET_KEY', 'ANTHROPIC_API_KEY'):
    try:
        ANTHROPIC_API_KEY = userdata.get(secret_name)
        if ANTHROPIC_API_KEY:
            print(f'API key loaded from Colab Secrets ({secret_name})')
            break
    except Exception:
        pass
if not ANTHROPIC_API_KEY:
    env_path = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(env_path):
        env = _parse_env(env_path)
        ANTHROPIC_API_KEY = (env.get('CHA_EXPERIMENT_SONNET_KEY')
                             or env.get('ANTHROPIC_API_KEY') or '')
        if ANTHROPIC_API_KEY:
            print(f'API key loaded from {env_path}')
        else:
            print(f'{env_path} found, but no CHA_EXPERIMENT_SONNET_KEY / ANTHROPIC_API_KEY '
                  f'in it. Keys present: {list(env)}')
    else:
        print(f'No .env at {env_path}. Folder contents: {sorted(os.listdir(PROJECT_DIR))[:25]}')
if ANTHROPIC_API_KEY:
    os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

print('='*64)
print(f'Project dir : {PROJECT_DIR}')
print(f'Checkpoints : {CKPT_DIR}   (Drive — model .pt files, downloadable)')
print(f'Pretrain    : {LOCAL_PT} (local, fast)  ↔  {DRIVE_PT} (Drive archive)')
print(f'API key     : {("..."+ANTHROPIC_API_KEY[-8:]) if ANTHROPIC_API_KEY else "NOT SET (Sonnet cells will be skipped)"}')
print('='*64)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
API key loaded from /content/drive/MyDrive/CA_Experiment_6/.env
Project dir : /content/drive/MyDrive/CA_Experiment_6
Checkpoints : /content/drive/MyDrive/CA_Experiment_6/checkpoints   (Drive — model .pt files, downloadable)
Pretrain    : /content/data/pretrain (local, fast)  ↔  /content/drive/MyDrive/CA_Experiment_6/data/pretrain (Drive archive)
API key     : ...9tYw8wAA


## Cell 2: Install dependencies & load Exp-6 assets

GPU torch (preinstalled on Colab) + tokenizers + datasets (free corpora) + anthropic (persona data + judge) + **qiskit/qiskit-aer + vaderSentiment** (the QPM, plan §5.5).

In [36]:
!pip install -q -U "tokenizers<=0.23.0" datasets anthropic python-dotenv qiskit qiskit-aer vaderSentiment tqdm

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import ca_assets as A
import qpm_bridge as QB
from model import ADA_80M, ADA_PILOT

print('special tokens :', A.SPECIAL_TOKENS)
print('dimensions     :', A.DIMENSIONS, '| probe turns', A.PROBE_TURNS)
print('ADA profile    :', QB.ADA_PROFILE)
_ps = QB.build_persona_state([QB.extract_d_vector('explain planck constant, I am unsure')])
print('QPM sanity     : source={source} register={register} ambivalence={ambivalence} certainty={certainty}'.format(**_ps))
print('-'*64)
print(f'Model (80M)    : d_model={ADA_80M.d_model} layers={ADA_80M.n_layers} heads={ADA_80M.n_heads} '
      f'vocab={ADA_80M.vocab_size} ctx={ADA_80M.max_seq_len} ffn={ADA_80M.hidden_dim()} head_dim={ADA_80M.head_dim()}')
print(f'Pilot          : d_model={ADA_PILOT.d_model} layers={ADA_PILOT.n_layers} (§5.4 gate)')

special tokens : ['<|bos|>', '<|eos|>', '<|pad|>', '<|system|>', '<|persona|>', '<|context|>', '<|user|>', '<|assistant|>', '<|endofturn|>']
dimensions     : ('T', 'E', 'C', 'S') | probe turns (5, 10, 15, 20, 25, 30, 35, 40)
ADA profile    : {'O_exp': 0.8, 'O_int': 0.85, 'O_val': 0.825, 'C_ind': 0.82, 'C_ord': 0.85, 'E_ent': 0.55, 'E_ass': 0.5, 'A_com': 0.68, 'A_pol': 0.8, 'N_vol': 0.1, 'N_wth': 0.12}
QPM sanity     : source=qpm register=professional_precise ambivalence=0.3831 certainty=0.2338
----------------------------------------------------------------
Model (80M)    : d_model=640 layers=12 heads=10 vocab=16000 ctx=1024 ffn=1728 head_dim=64
Pilot          : d_model=256 layers=6 (§5.4 gate)


## Cell 3: Verify GPU & report throughput expectations

In [39]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Change runtime type to a GPU (A100/L4 for the full run; T4 = smoke only).')

p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 1e9
print(f'GPU            : {p.name}')
print(f'VRAM           : {vram:.1f} GB')
print(f'bf16 supported : {torch.cuda.is_bf16_supported()}')
print(f'compute cap    : {p.major}.{p.minor}')

# Rough Stage-A ETA for 2B tokens by GPU class (plan §5.3).
name = p.name.lower()
if 'a100' in name:      eta = '~11 h (~50k tok/s)'
elif 'l4' in name:      eta = '~20-25 h (~25k tok/s)'
elif 't4' in name:      eta = '~50 h — SMOKE/PILOT ONLY (§5.3)'
else:                   eta = 'unknown — benchmark with the throughput logs'
print(f'Stage-A 2B tok : {eta}')
if vram < 16:
    print('WARNING: <16 GB VRAM — reduce --batch-size / --seq-len; the 80M model itself fits, but keep an eye on OOM.')
print('Exp 6 GPU environment: READY')

CUDA available : True
GPU            : NVIDIA A100-SXM4-80GB
VRAM           : 85.1 GB
bf16 supported : True
compute cap    : 8.0
Stage-A 2B tok : ~11 h (~50k tok/s)
Exp 6 GPU environment: READY


## Cell 4: Anthropic API smoke test

Confirms the key works before the persona-data / judging cells. Skipped (not fatal) if no key is set.

In [38]:
GEN_MODEL   = 'claude-sonnet-4-6'   # persona/style/refusal data (plan §4.2); $3/$15 per 1M tok
JUDGE_MODEL = 'claude-sonnet-4-5'   # judge id matches Exp 1-5; $3/$15 per 1M tok

if os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    import anthropic
    _client = anthropic.Anthropic()
    _r = _client.messages.create(model=JUDGE_MODEL, max_tokens=10,
                                 messages=[{'role': 'user', 'content': 'Say "ok".'}])
    print(f'API smoke test : {_r.content[0].text.strip()}  (gen={GEN_MODEL}, judge={JUDGE_MODEL})')
else:
    print('No API key — the Sonnet persona-data (P1d) and judging (P5) cells will be skipped.')

API smoke test : ok  (gen=claude-sonnet-4-6, judge=claude-sonnet-4-5)


---
## P0 · Provision — free Stage-A corpus text (for the tokenizer)

Stream a slice of FineWeb-Edu (or offline Wikipedia/ZIM) to a text file the BPE trains on. FREE (plan §4.2). Resumable — skips if the sample exists.

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import time

SAMPLE_TXT     = 'data/pretrain/corpus_sample.txt'
TOKENIZER_DOCS = 200_000   # docs used to TRAIN the BPE (raise for a richer vocab)

if not Path(SAMPLE_TXT).exists():
    t0 = time.time()
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    n_chars = 0
    with open(SAMPLE_TXT, 'w') as f:
        for i, ex in enumerate(tqdm(ds, total=TOKENIZER_DOCS, desc='corpus sample')):
            if i >= TOKENIZER_DOCS:
                break
            txt = ex['text'].replace('\n', ' ')
            n_chars += len(txt)
            f.write(txt + '\n')
    print(f'wrote {SAMPLE_TXT}: {TOKENIZER_DOCS:,} docs, {n_chars/1e6:.1f} M chars in {time.time()-t0:.0f}s')
else:
    print(f'{SAMPLE_TXT} exists ({os.path.getsize(SAMPLE_TXT)/1e6:.1f} MB) — skipping')

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

corpus sample: 100%|██████████| 200000/200000 [00:27<00:00, 7224.05it/s]

wrote data/pretrain/corpus_sample.txt: 200,000 docs, 947.8 M chars in 34s


## P1a · Train the 16k BPE tokenizer (+ ADA/QPM special tokens)

In [ ]:
if not Path('tokenizer/ada_bpe.json').exists():
    !python train_tokenizer.py --input data/pretrain/corpus_sample.txt --vocab-size 16000 --out tokenizer/ada_bpe.json
else:
    print('tokenizer exists — skipping')

from tokenizer_util import ADATokenizer
tok = ADATokenizer.load()
print(f'tokenizer      : vocab={tok.vocab_size}  persona_id={tok.persona_id}  assistant_id={tok.assistant_id}  eot_id={tok.eot_id}')
_ids = tok.encode('Tungsten melts at 3422 C.')
print(f'roundtrip test : {_ids[:8]}... -> {tok.decode(_ids)!r}')

Training BPE on 1 file(s) → vocab 16000
[00:00:01] Tokenize words                 ██████████████████ 1313063  /  1313063
[00:00:01] Count pairs                    ██████████████████ 1313063  /  1313063
[00:00:07] Compute merges                 ██████████████████ 15735    /    15735
Saved tokenizer → tokenizer/ada_bpe.json  (vocab_size=16000)
  <|bos|>          -> id 0
  <|eos|>          -> id 1
  <|pad|>          -> id 2
  <|system|>       -> id 3
  <|persona|>      -> id 4
  <|context|>      -> id 5
  <|user|>         -> id 6
  <|assistant|>    -> id 7
  <|endofturn|>    -> id 8
tokenizer      : vocab=16000  persona_id=4  assistant_id=7  eot_id=8
roundtrip test : [60, 2047, 315, 276, 4745, 794, 420, 8356]... -> 'Tungsten melts at 3422 C.'


## P1b · Tokenize Stage-A corpus → uint16 bins (local disk + Drive archive)

Target 1–3 B tokens (plan §5.2). Bins are built on **local SSD** (`LOCAL_PT`) for fast training, then copied **once** to the **Drive archive** (`DRIVE_PT`) so a durable, downloadable copy survives disconnects. On reconnect this cell restores from the archive instead of re-tokenizing. The model weights are separate — they live as `.pt` checkpoints under `CKPT_DIR` on Drive.

In [ ]:

import shutil

def _have(d):
    return os.path.exists(f'{d}/train.bin') and os.path.exists(f'{d}/val.bin')

if _have(LOCAL_PT):
    print(f'local pretrain bins present in {LOCAL_PT} — using them')
elif _have(DRIVE_PT):
    print(f'restoring pretrain bins from Drive archive → {LOCAL_PT} (sequential copy)...')
    for f in ('train.bin', 'val.bin'):
        shutil.copy(f'{DRIVE_PT}/{f}', f'{LOCAL_PT}/{f}')
    print('restored from archive (no re-tokenization needed)')
else:
    # Build on local disk (fast), reading the tokenizer from Drive.
    !python prepare_data.py pretrain --hf-dataset HuggingFaceFW/fineweb-edu --hf-config sample-10BT \
        --text-field text --out-dir $LOCAL_PT --max-tokens 2000000000
    print(f'archiving bins to Drive → {DRIVE_PT} (one-time sequential copy, downloadable) ...')
    for f in ('train.bin', 'val.bin'):
        shutil.copy(f'{LOCAL_PT}/{f}', f'{DRIVE_PT}/{f}')
    print('archived to Drive')

import numpy as np
for split in ('train', 'val'):
    b = Path(f'{LOCAL_PT}/{split}.bin')
    if b.exists():
        print(f'{split}.bin : {b.stat().st_size//2:,} tokens ({b.stat().st_size/1e6:.0f} MB, local)')

restoring pretrain bins from Drive archive → /content/data/pretrain (sequential copy)...
restored from archive (no re-tokenization needed)
train.bin : 1,990,001,553 tokens (3980 MB, local)
val.bin : 10,000,007 tokens (20 MB, local)


## P1c · Free reading-skill QA + held-out eval sets

SQuAD2 backbone (unanswerables supervise H2 abstention) → §4.3 records. Eval sets carry gold answers for EM/F1. FREE.

In [ ]:
if not Path('data/qa_sft.jsonl').exists():
    !python prepare_data.py qa-sft --out data/qa_sft.jsonl        # SQuAD2 train → reading skill
if not Path('data/eval_answerable.jsonl').exists():
    !python prepare_data.py eval --n 200                          # held-out answerable + unanswerable

import json, collections
cnt = collections.Counter(json.loads(l)['source'] for l in open('data/qa_sft.jsonl'))
print('qa_sft.jsonl by source :', dict(cnt), '| total', sum(cnt.values()))
print('eval_answerable        :', sum(1 for _ in open('data/eval_answerable.jsonl')))
print('eval_unanswerable      :', sum(1 for _ in open('data/eval_unanswerable.jsonl')))






qa-sft: 130319 records → data/qa_sft.jsonl  (answerable 86821, unanswerable 43498)
eval: 200 answerable → data/eval_answerable.jsonl; 200 unanswerable → data/eval_unanswerable.jsonl
qa_sft.jsonl by source : {'squad2': 130319} | total 130319
eval_answerable        : 200
eval_unanswerable      : 200


## P1d · ADA persona layer (Sonnet, bounded) — QPM-conditioned

Each persona/style/refusal record is tagged with a QPM `persona_state` (marginals + valence + register + ambivalence/certainty) from the user turns' d-vectors; the affect directive is fed to the teacher so the target reflects it (plan §5.5). Bounded by `--budget`; raw archived to `data/raw_sonnet/`.

In [ ]:
if os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    import json, collections

    def _count(src):
        if not os.path.exists('data/qa_sft.jsonl'):
            return 0
        return sum(1 for l in open('data/qa_sft.jsonl') if json.loads(l).get('source') == src)

    n_persona = _count('sonnet_persona')
    n_style   = _count('sonnet_style')
    n_refusal = _count('sonnet_refusal')
    n_scripts = len(list(Path('data/persona_scripts').glob('script_*.json')))
    need_persona = max(0, 40 - n_persona)
    need_style   = max(0, 40 - n_style)
    need_refusal = max(0, 30 - n_refusal)
    print(f'have: persona={n_persona} style={n_style} refusal={n_refusal} scripts={n_scripts}')

    # Persona conversations (multi-turn, episodic callbacks + capability-edge abstention)
    if need_persona:
        !python gen_persona_data.py persona --model $GEN_MODEL --budget {need_persona} --out data/qa_sft.jsonl
    else:
        print('persona: complete — skipping')

    # ADA-voice restyling of a slice of the free QA answers
    if need_style:
        !python gen_persona_data.py style --model $GEN_MODEL --budget {need_style} --inputs data/qa_sft.jsonl --out data/qa_sft.jsonl
    else:
        print('style: complete — skipping')

    # Refusal phrasing set (near-miss / off-topic / empty context)
    if need_refusal:
        !python gen_persona_data.py refusal --model $GEN_MODEL --budget {need_refusal} --out data/qa_sft.jsonl
    else:
        print('refusal: complete — skipping')

    # ~20 PersonaScore eval conversation scripts (40 user turns each)
    if n_scripts < 20:
        !python gen_persona_data.py eval-scripts --model $GEN_MODEL --n-scripts 20 --n-turns 40
    else:
        print('eval-scripts: complete — skipping')

    cnt = collections.Counter(json.loads(l)['source'] for l in open('data/qa_sft.jsonl'))
    print('qa_sft by source :', dict(cnt))
    print('persona scripts  :', len(list(Path('data/persona_scripts').glob('script_*.json'))))
else:
    print('No API key — skipping Sonnet persona data. (Free reading-skill SFT still trains persona-thin.)')


have: persona=40 style=40 refusal=30 scripts=5
persona: complete — skipping
style: complete — skipping
refusal: complete — skipping
  [  5/20] eval-script 'human biology and anatomy' — 40 turns
  [  7/20] eval-script 'famous mathematicians and theore' — 40 turns
  [  8/20] eval-script 'materials science and metals' — 41 turns
  [  9/20] eval-script 'weather and climate basics' — 40 turns
  [ 10/20] eval-script 'the periodic table' — 40 turns
  [ 11/20] eval-script 'renewable energy technologies' — 40 turns
  [ 12/20] eval-script 'notable space missions' — 40 turns
  [ 13/20] eval-script 'classical mechanics' — 40 turns
  [ 14/20] eval-script 'cell biology fundamentals' — 40 turns
  [ 15/20] eval-script 'programming language history' — 40 turns
  [ 16/20] eval-script 'ancient civilizations' — 40 turns
  [ 17/20] eval-script 'oceans and marine life' — 40 turns
  [ 18/20] eval-script 'the human immune system' — 40 turns
  [ 19/20] eval-script 'electricity and magnetism' — 40 turns
  [ 20/

In [ ]:
import json, collections
from pathlib import Path
import ca_assets as A
from tokenizer_util import ADATokenizer
from data_utils import tokenize_sft_record

print('='*60, '\nVERIFY — Exp 6 data provisioning\n' + '='*60)

# 1. Tokenizer (16k BPE + special tokens incl. <|persona|>)
tok = ADATokenizer.load()
print(f'[tokenizer] vocab={tok.vocab_size}  persona_id={tok.persona_id}  eot_id={tok.eot_id}')

# 2. Pretrain bins (local working copy + Drive archive)
for tag, d in [('local', LOCAL_PT), ('drive', DRIVE_PT)]:
    for sp in ('train', 'val'):
        p = Path(f'{d}/{sp}.bin')
        print(f'[pretrain:{tag:5}] {sp}.bin: ' + (f'{p.stat().st_size/1e6:.0f} MB, {p.stat().st_size//2:,} tokens' if p.exists() else 'MISSING'))

# 3. SFT data: source counts (expect squad2 ~130k, persona 40, style ~40, refusal 30)
recs = [json.loads(l) for l in open('data/qa_sft.jsonl')]
cnt = collections.Counter(r['source'] for r in recs)
print(f'[qa_sft] total={len(recs)}  by_source={dict(cnt)}')

# 4. QPM channel: persona_state present on every sonnet_* record
sonnet = [r for r in recs if r['source'].startswith('sonnet_')]
with_ps = sum(1 for r in sonnet if r.get('persona_state'))
print(f'[QPM] sonnet records with persona_state: {with_ps}/{len(sonnet)}')

# 5. Schema validation (all sonnet + a squad2 sample)
bad = []
for r in sonnet + recs[:500]:
    try: A.validate_record(r)
    except Exception as e: bad.append((r.get('id'), str(e)[:50]))
print(f'[schema] validation errors: {len(bad)}' + (f'  e.g. {bad[:2]}' if bad else ''))

# 6. Eval sets (expect ~200 each, answerable carry gold_answers)
for f in ('eval_answerable', 'eval_unanswerable'):
    p = Path(f'data/{f}.jsonl')
    n = sum(1 for _ in open(p)) if p.exists() else 0
    gold = sum(1 for l in open(p) if json.loads(l).get('gold_answers')) if p.exists() and f.endswith('answerable') else '-'
    print(f'[eval] {f}: {n}  (with gold: {gold})')

# 7. PersonaScore scripts (expect 20, each 40 turns w/ contexts)
scripts = sorted(Path('data/persona_scripts').glob('script_*.json'))
tc = [len(json.load(open(s)).get('turns', [])) for s in scripts]
print(f'[scripts] count={len(scripts)}  turns/script: min={min(tc) if tc else 0} max={max(tc) if tc else 0}')

# 8. SFT tokenization sanity — a persona record must produce a supervised span + the <|persona|> channel
ex = next((tokenize_sft_record(r, tok, 1024) for r in sonnet if r['source']=='sonnet_persona'), None)
if ex:
    print(f'[sft] persona example: {len(ex["input_ids"])} toks | supervised={sum(ex["loss_mask"])} | persona_channel={tok.persona_id in
ex["input_ids"]}')
print('='*60)

VERIFY — Exp 6 data provisioning
[tokenizer] vocab=16000  persona_id=4  eot_id=8
[pretrain:local] train.bin: 3980 MB, 1,990,001,553 tokens
[pretrain:local] val.bin: 20 MB, 10,000,007 tokens
[pretrain:drive] train.bin: 3980 MB, 1,990,001,553 tokens
[pretrain:drive] val.bin: 20 MB, 10,000,007 tokens
[qa_sft] total=130429  by_source={'squad2': 130319, 'sonnet_persona': 40, 'sonnet_refusal': 30, 'sonnet_style': 40}
[QPM] sonnet records with persona_state: 110/110
[schema] validation errors: 0
[eval] eval_answerable: 200  (with gold: 200)
[eval] eval_unanswerable: 200  (with gold: 0)
[scripts] count=20  turns/script: min=40 max=43
[sft] persona example: 1024 toks | supervised=577 | persona_channel=True


---
## Pilot gate (plan §5.4) — does the pipeline learn at all?

Tiny model (d_model 256, 6 layers) on a small slice + the SFT data, on any GPU. Confirm loss falls and the assistant span is learned **before** spending A100 hours. Detailed loss/throughput logs stream from the training scripts.

In [ ]:
!python train_pretrain.py --config pilot --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin \
    --ckpt-dir $CKPT_DIR --max-steps 2000 --batch-size 16 --grad-accum 4 --seq-len 512 --ckpt-interval 500
!python train_sft.py --init $CKPT_DIR/pretrain_pilot_final.pt --sft data/qa_sft.jsonl \
    --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR --max-steps 800 --seq-len 512

config=pilot device=cuda dtype=torch.bfloat16 seq_len=512
=== Stage-A pretrain — config=pilot ===
    params 8.9M | d_model 256 layers 6 heads 8 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 2000 | tok/step 32,768 | target tokens 0.07B
step      0/2000 | loss 9.7240 | lr 6.00e-07 | gnorm 2.58 | 52.6k tok/s | seen 0M | ETA 0.3h
step     20/2000 | loss 9.6543 | lr 1.26e-05 | gnorm 2.64 | 357.0k tok/s | seen 1M | ETA 0.1h
step     40/2000 | loss 9.3872 | lr 2.46e-05 | gnorm 1.35 | 356.7k tok/s | seen 1M | ETA 0.1h
step     60/2000 | loss 9.1993 | lr 3.66e-05 | gnorm 1.27 | 357.7k tok/s | seen 2M | ETA 0.0h
step     80/2000 | loss 8.9825 | lr 4.86e-05 | gnorm 1.25 | 353.5k tok/s | seen 3M | ETA 0.0h
step    100/2000 | loss 8.7112 | lr 6.06e-05 | gnorm 1.26 | 352.2k tok/s | seen 3M | ETA 0.0h
step    120/2000 | loss 8.4427 | lr 7.26e-05 | gnorm 1.19 | 354.7k tok/s | seen 4M | ETA 0.0h
step    140/2000 | loss 8.1738 | lr 8.46e-05 | gnorm 1.15 | 357.9k tok/s | seen 5M | ETA 0

In [ ]:
# Pilot eyeball: grounded answer + abstention + QPM persona channel live (plan §5.4 gate)
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED  Q: melting point of tungsten?')
print('          A:', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN   Q: boiling point of nitrogen? (not in context)')
print('          A:', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('\nPilot gate: PASS if loss fell, grounded answer is on-topic, and abstention fires. Only then run P2.')

GROUNDED  Q: melting point of tungsten?
          A: I don't have data related to this.
ABSTAIN   Q: boiling point of nitrogen? (not in context)
          A: The water is a high-mile-tree-fifth-six-military, and the water-six-military-gall-military-gall-six-military-gall-s-s-military-gall-s-military-s-s-s-s-mil-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-s-

Pilot gate: PASS if loss fell, grounded answer is on-topic, and abstention fires. Only then run P2.


---
## P2 · Stage-A pretrain (~80M, A100)

Next-token LM, cosine LR + warmup, bf16, checkpoint-to-Drive every 500 steps. **Resumable** — re-run this cell with `--resume` after a disconnect. Stops on val-loss plateau. Watch the `tok/s` and `val_loss` logs.

In [ ]:
!python train_pretrain.py --config 80m --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin \
    --ckpt-dir $CKPT_DIR --max-steps 40000 --batch-size 32 --grad-accum 8 --seq-len 1024 \
    --ckpt-interval 500 --resume

import glob
print('Stage-A checkpoints on Drive:')
for f in sorted(glob.glob(f'{CKPT_DIR}/pretrain_80m*.pt'))[-5:]:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)')

config=80m device=cuda dtype=torch.bfloat16 seq_len=1024
=== Stage-A pretrain — config=80m ===
    params 69.7M | d_model 640 layers 12 heads 10 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 40000 | tok/step 262,144 | target tokens 10.49B
resumed from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_last.pt @ step 22001
step  22020/40000 | loss 2.9772 | lr 1.46e-04 | gnorm 0.38 | 156.3k tok/s | seen 5773M | ETA 8.4h
step  22040/40000 | loss 2.9842 | lr 1.46e-04 | gnorm 0.36 | 160.2k tok/s | seen 5778M | ETA 8.2h
step  22060/40000 | loss 2.9788 | lr 1.46e-04 | gnorm 0.38 | 159.8k tok/s | seen 5783M | ETA 8.2h
step  22080/40000 | loss 2.9832 | lr 1.45e-04 | gnorm 0.40 | 159.7k tok/s | seen 5788M | ETA 8.2h
step  22100/40000 | loss 2.9753 | lr 1.45e-04 | gnorm 0.39 | 159.7k tok/s | seen 5794M | ETA 8.2h
step  22120/40000 | loss 2.9787 | lr 1.45e-04 | gnorm 0.39 | 159.6k tok/s | seen 5799M | ETA 8.2h
step  22140/40000 | loss 2.9605 | lr 1.45e-04 | gnorm 0.39

## P3 · Stage-B SFT (grounded-QA + ADA persona/style/refusal + QPM channel)

Assistant-span-masked loss; Stage-A replay to prevent fluency forgetting; the `<|persona|>` QPM channel is present in persona-bearing examples.

In [ ]:
!python train_sft.py --init $CKPT_DIR/pretrain_80m_best.pt --sft data/qa_sft.jsonl \
    --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR \
    --max-steps 6000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 --resume
print('SFT model:', f'{CKPT_DIR}/sft_final.pt', '(', os.path.getsize(f'{CKPT_DIR}/sft_final.pt')/1e6, 'MB )')

SFTDataset: 130429 examples (0 skipped) from data/qa_sft.jsonl
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | params 69.7M | device cuda
=== Stage-B SFT — 130429 examples (89 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.1 ===
    max_steps 6000 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/6000 | loss 4.9326 | lr 1.00e-06 | gnorm 38.55 | replay 1 | 0.98s/step | ETA 98.3m
step    20/6000 | loss 3.0406 | lr 2.10e-05 | gnorm 7.05 | replay 12 | 0.45s/step | ETA 45.1m
step    40/6000 | loss 1.2185 | lr 4.10e-05 | gnorm 4.36 | replay 18 | 0.45s/step | ETA 45.0m
step    60/6000 | loss 1.2512 | lr 6.10e-05 | gnorm 3.82 | replay 25 | 0.45s/step | ETA 44.8m
step    80/6000 | loss 1.1726 | lr 8.10e-05 | gnorm 3.47 | replay 31 | 0.45s/step | ETA 44.8m
step   100/6000 | loss 1.1575 | lr 1.00e-04 | gnorm 3.55 | replay 37 | 0.45s/step | ETA 44.6m
step   120/6000 | loss 1.2458 | lr 1.00e-04 | gnorm 3.26 | replay 45 | 0.45s/step | ETA 44.4m


## P4 · Fallback baseline (RQ5 / H5, optional)

Fine-tune a small **pretrained** base (SmolLM2-135M) on the identical Stage-B data so the fallback is a one-step swap. Run only if you want the H5 gap now; skip to save compute. *(Uses `transformers`; add it to the install cell.)*

In [ ]:
# Placeholder for the RQ5 baseline: fine-tune HuggingFaceTB/SmolLM2-135M on data/qa_sft.jsonl
# (reformat records to the base's chat template; run §6.1-6.2 on it). The corpus/pipeline are
# reused unchanged — only the base swaps. Left as a documented optional step (plan §6.4).
print('RQ5 fallback baseline: optional — see plan §6.4 / decision-rule H1-fail row.')

---
## P5 · Evaluation

### QA competence — H1 (correct-and-grounded + EM/F1) and H2 (abstention F1 + hallucination)

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt \
    --answerable data/eval_answerable.jsonl --unanswerable data/eval_unanswerable.jsonl \
    --out results/qa_results.json

import json
m = json.load(open('results/qa_results.json'))['metrics']
print('='*56)
print(f"H1 correct-and-grounded : {m['H1_correct_grounded_rate']:.3f}  (>=0.70 -> {m['H1_pass']})")
print(f"   SQuAD2 EM / F1        : {m['squad_EM']} / {m['squad_F1']}")
print(f"H2 abstention F1        : {m['H2_abstention_F1']:.3f}  (>=0.80 -> {m['H2_pass']})")
print(f"   hallucination rate   : {m['hallucination_rate']}")
print(f"   over-refusal rate    : {m['over_refusal_rate']}")
print('='*56)

=== QA eval (H1/H2) | ckpt=sft_final.pt device=cuda QPM=ON judge=claude-sonnet-4-5 ===
    answerable=200  unanswerable=200
  answerable [  10/200]  running H1=0.00  over-refusal=0.00  ETA  980s
  answerable [  20/200]  running H1=0.00  over-refusal=0.00  ETA  879s
  answerable [  30/200]  running H1=0.00  over-refusal=0.00  ETA  835s
  answerable [  40/200]  running H1=0.00  over-refusal=0.00  ETA  796s
  answerable [  50/200]  running H1=0.00  over-refusal=0.00  ETA  750s
  answerable [  60/200]  running H1=0.00  over-refusal=0.00  ETA  697s
  answerable [  70/200]  running H1=0.00  over-refusal=0.00  ETA  646s
  answerable [  80/200]  running H1=0.00  over-refusal=0.00  ETA  584s
  answerable [  90/200]  running H1=0.00  over-refusal=0.00  ETA  528s
  answerable [ 100/200]  running H1=0.01  over-refusal=0.00  ETA  483s
  answerable [ 110/200]  running H1=0.01  over-refusal=0.00  ETA  438s
  answerable [ 120/200]  running H1=0.01  over-refusal=0.00  ETA  390s
  answerable [ 130/200] 

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt \
    --answerable data/eval_answerable.jsonl --unanswerable data/eval_unanswerable.jsonl \
    --no-qpm --out results/qa_results_noqpm.json

import json
m = json.load(open('results/qa_results_noqpm.json'))['metrics']
print('='*56)
print(f"H1 correct-and-grounded : {m['H1_correct_grounded_rate']:.3f}  (>=0.70 -> {m['H1_pass']})")
print(f"   SQuAD2 EM / F1        : {m['squad_EM']} / {m['squad_F1']}")
print(f"H2 abstention F1        : {m['H2_abstention_F1']:.3f}  (>=0.80 -> {m['H2_pass']})")
print(f"   hallucination rate   : {m['hallucination_rate']}")
print(f"   over-refusal rate    : {m['over_refusal_rate']}")
print('='*56)

=== QA eval (H1/H2) | ckpt=sft_final.pt device=cuda QPM=OFF judge=claude-sonnet-4-5 ===
    answerable=200  unanswerable=200
  answerable [  10/200]  running H1=0.10  over-refusal=0.10  ETA  883s
  answerable [  20/200]  running H1=0.10  over-refusal=0.05  ETA  848s
  answerable [  30/200]  running H1=0.07  over-refusal=0.03  ETA  805s
  answerable [  40/200]  running H1=0.05  over-refusal=0.03  ETA  787s
  answerable [  50/200]  running H1=0.04  over-refusal=0.02  ETA  742s
  answerable [  60/200]  running H1=0.03  over-refusal=0.03  ETA  671s
  answerable [  70/200]  running H1=0.03  over-refusal=0.04  ETA  611s
  answerable [  80/200]  running H1=0.03  over-refusal=0.04  ETA  557s
  answerable [  90/200]  running H1=0.02  over-refusal=0.03  ETA  518s
  answerable [ 100/200]  running H1=0.02  over-refusal=0.03  ETA  471s
  answerable [ 110/200]  running H1=0.02  over-refusal=0.03  ETA  423s
  answerable [ 120/200]  running H1=0.02  over-refusal=0.03  ETA  376s
  answerable [ 130/200]

In [ ]:
import json
r = json.load(open('results/qa_results_noqpm.json'))
print('--- ANSWERABLE (should answer from context) ---')
for it in r['items']['answerable'][:5]:
    print('Q:', it['question']); print('gold:', it['gold'])
    print('A:', repr(it['response'][:200]))
    print('  correct_grounded:', it['correct_and_grounded'], '| abstained:', it['abstained'], '\n')
print('--- UNANSWERABLE (should abstain) ---')
for it in r['items']['unanswerable'][:5]:
    print('Q:', it['question']); print('A:', repr(it['response'][:200]))
    print('  abstained:', it['abstained'], '\n')

--- ANSWERABLE (should answer from context) ---
Q: In what country is Normandy located?
gold: ['France', 'France', 'France', 'France']
A: "In the 11th century, the Normans were the people who lived in the region of the Viking Kingdom, where they were said to be descended from a common ancestor of the Viking Kingdom's peoples. In the 11th"
  correct_grounded: False | abstained: False 

Q: When were the Normans in Normandy?
gold: ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries']
A: 'The Normans were the people who in the 10th century brought the language of the Normans to the English-speaking world. They were the people who had been the first to use the language of the Normans in'
  correct_grounded: False | abstained: False 

Q: From which countries did the Norse originate?
gold: ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway']
A: "I don't have da

In [ ]:
# free threshold sweep (no API cost; --dry-run-judge skips the Sonnet judge,
# and EM/F1/H2/hallucination are all judge-free)
for tau in [0.0, 0.05, 0.1, 0.2, 0.3]:
  print(f'\n===== abstain_threshold = {tau} =====')
  !python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt --constrained --no-qpm \
    --abstain-threshold {tau} --dry-run-judge --out /tmp/qa_c_{tau}.json 2>&1 \
    | grep -E "squad_EM|squad_F1|H2_abstention_F1|hallucination_rate|over_refusal_rate"


===== abstain_threshold = 0.0 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0813,
  "H2_abstention_F1": 0.251,
  "hallucination_rate": 0.84,
  "over_refusal_rate": 0.115,

===== abstain_threshold = 0.05 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0367,
  "H2_abstention_F1": 0.514,
  "hallucination_rate": 0.45,
  "over_refusal_rate": 0.59,

===== abstain_threshold = 0.1 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0192,
  "H2_abstention_F1": 0.5613,
  "hallucination_rate": 0.29,
  "over_refusal_rate": 0.82,

===== abstain_threshold = 0.2 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0091,
  "H2_abstention_F1": 0.623,
  "hallucination_rate": 0.12,
  "over_refusal_rate": 0.945,

===== abstain_threshold = 0.3 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0059,
  "H2_abstention_F1": 0.661,
  "hallucination_rate": 0.03,
  "over_refusal_rate": 0.965,


In [ ]:
!pip install -q sentence-transformers
for rr in ['', '--rerank --rerank-topk 1', '--rerank --rerank-topk 2']:
      for tau in [0.0, 0.1]:
          print(f'\n===== rerank="{rr or "off"}"  τ={tau} =====')
          !python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt --constrained --no-qpm \
              {rr} --abstain-threshold {tau} --dry-run-judge --out /tmp/rr.json 2>&1 \
              | grep -E "squad_EM|squad_F1|H2_abstention_F1|hallucination_rate|over_refusal_rate"



===== rerank="off"  τ=0.0 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0813,
  "H2_abstention_F1": 0.251,
  "hallucination_rate": 0.84,
  "over_refusal_rate": 0.115,

===== rerank="off"  τ=0.1 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0192,
  "H2_abstention_F1": 0.5613,
  "hallucination_rate": 0.29,
  "over_refusal_rate": 0.82,

===== rerank="--rerank --rerank-topk 1"  τ=0.0 =====
  "squad_EM": 0.0,
  "squad_F1": 0.1427,
  "H2_abstention_F1": 0.2548,
  "hallucination_rate": 0.835,
  "over_refusal_rate": 0.13,

===== rerank="--rerank --rerank-topk 1"  τ=0.1 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0357,
  "H2_abstention_F1": 0.591,
  "hallucination_rate": 0.245,
  "over_refusal_rate": 0.8,

===== rerank="--rerank --rerank-topk 2"  τ=0.0 =====
  "squad_EM": 0.0,
  "squad_F1": 0.1048,
  "H2_abstention_F1": 0.278,
  "hallucination_rate": 0.82,
  "over_refusal_rate": 0.115,

===== rerank="--rerank --rerank-topk 2"  τ=0.1 =====
  "squad_EM": 0.0,
  "squad_F1": 0.0263,
  "H2_abstention_F1": 0.581

In [ ]:
!python prepare_data.py qa-sft --keep-sonnet --out data/qa_sft.jsonl
import json, collections
recs=[json.loads(l) for l in open('data/qa_sft.jsonl')]
print(collections.Counter(r['source'] for r in recs))     # expect sonnet_* still 40/40/30
ans=[r for r in recs if r['source']=='squad2' and r['answerable']][0]
print('sample target:', next(m['content'] for m in ans['messages'] if m['role']=='assistant'))  # bare span, no suffix

keep-sonnet: preserving 110 existing Sonnet records





qa-sft: 130319 reading records + 110 preserved Sonnet → data/qa_sft.jsonl  (reading answerable 86821, unanswerable 43498)
Counter({'squad2': 130319, 'sonnet_persona': 40, 'sonnet_style': 40, 'sonnet_refusal': 30})
sample target: in the late 1990s


In [ ]:
!python train_sft.py --init $CKPT_DIR/pretrain_80m_best.pt --sft data/qa_sft.jsonl \
      --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR \
      --max-steps 6000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.05

SFTDataset: 130429 examples (0 skipped) from data/qa_sft.jsonl
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | params 69.7M | device cuda
=== Stage-B SFT — 130429 examples (89 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.05 ===
    max_steps 6000 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/6000 | loss 6.2321 | lr 1.00e-06 | gnorm 74.27 | replay 0 | 1.50s/step | ETA 150.1m
step    20/6000 | loss 3.7364 | lr 2.10e-05 | gnorm 14.91 | replay 6 | 0.45s/step | ETA 45.0m
step    40/6000 | loss 1.8592 | lr 4.10e-05 | gnorm 7.17 | replay 8 | 0.45s/step | ETA 45.0m
step    60/6000 | loss 1.8367 | lr 6.10e-05 | gnorm 5.98 | replay 11 | 0.45s/step | ETA 44.8m
step    80/6000 | loss 1.6210 | lr 8.10e-05 | gnorm 10.96 | replay 12 | 0.45s/step | ETA 44.7m
step   100/6000 | loss 1.6307 | lr 1.00e-04 | gnorm 5.47 | replay 15 | 0.45s/step | ETA 44.4m
step   120/6000 | loss 1.6774 | lr 1.00e-04 | gnorm 3.69 | replay 19 | 0.45s/step | ETA 44.3

In [ ]:
for rr in ['', '--rerank --rerank-topk 1']:
      for tau in [0.0, 0.1]:
          print(f'\n=== rerank={rr or "off"} τ={tau} ===')
          !python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt --constrained --no-qpm \
              {rr} --abstain-threshold {tau} --dry-run-judge --out /tmp/rr.json 2>&1 \
              | grep -E "squad_EM|squad_F1|H2_abstention_F1|hallucination_rate|over_refusal_rate"


=== rerank=off τ=0.0 ===
  "squad_EM": 0.005,
  "squad_F1": 0.0962,
  "H2_abstention_F1": 0.1983,
  "hallucination_rate": 0.88,
  "over_refusal_rate": 0.09,

=== rerank=off τ=0.1 ===
  "squad_EM": 0.0,
  "squad_F1": 0.0183,
  "H2_abstention_F1": 0.5885,
  "hallucination_rate": 0.235,
  "over_refusal_rate": 0.835,

=== rerank=--rerank --rerank-topk 1 τ=0.0 ===
  "squad_EM": 0.0,
  "squad_F1": 0.1575,
  "H2_abstention_F1": 0.2985,
  "hallucination_rate": 0.8,
  "over_refusal_rate": 0.14,

=== rerank=--rerank --rerank-topk 1 τ=0.1 ===
  "squad_EM": 0.0,
  "squad_F1": 0.0342,
  "H2_abstention_F1": 0.6107,
  "hallucination_rate": 0.2,
  "over_refusal_rate": 0.82,


In [ ]:
import json
r=json.load(open('/tmp/rr.json'))
for it in r['items']['answerable'][:5]:
    print('Q:', it['question'], '| gold:', it['gold'], '| A:', repr(it['response']), '| EM:', it['em'])

Q: In what country is Normandy located? | gold: ['France', 'France', 'France', 'France'] | A: 'Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy,' | EM: 0.0
Q: When were the Normans in Normandy? | gold: ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries'] | A: 'Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy,' | EM: 0.0
Q: From which countries did the Norse originate? | gold: ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway'] | A: "I don't have data related to this." | EM: 0.0
Q: Who was the Norse leader? | gold: ['Rollo', 'Rollo', 'Rollo', 'Rollo'] | A: 'Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fe

In [ ]:
# try free decoder again
for rr in ['', '--rerank --rerank-topk 1']:
    print(f'\n=== FREE decode  rerank={rr or "off"} ===')
    !python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt --no-qpm {rr} \
        --dry-run-judge --out /tmp/free.json 2>&1 \
        | grep -E "squad_EM|squad_F1|hallucination_rate|over_refusal_rate"
import json
r=json.load(open('/tmp/free.json'))
for it in r['items']['answerable'][:5]:
    print('gold:', it['gold'], '| A:', repr(it['response'][:80]), '| EM:', it['em'])


=== FREE decode  rerank=off ===
  "squad_EM": 0.0,
  "squad_F1": 0.0142,
  "hallucination_rate": 0.965,
  "over_refusal_rate": 0.015,

=== FREE decode  rerank=--rerank --rerank-topk 1 ===
  "squad_EM": 0.0,
  "squad_F1": 0.0208,
  "hallucination_rate": 0.96,
  "over_refusal_rate": 0.035,
gold: ['France', 'France', 'France', 'France'] | A: 'Normandy was the site of the first invasion of Normandy in 1066. In 1066, Norman' | EM: 0.0
gold: ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries'] | A: 'N Normandy was the people who in the 10th century BC gave their name to the regi' | EM: 0.0
gold: ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway'] | A: "Scotland's origin is from the Norse language, but the context for the passage st" | EM: 0.0
gold: ['Rollo', 'Rollo', 'Rollo', 'Rollo'] | A: 'Norsehanishad, the son of King Charles II of Norse, was the first t

In [ ]:
# Symbolic retriever eval — the head-to-head vs the 80M reader
!pip install -q spacy sentence-transformers
!python -m spacy download en_core_web_sm

for tau in [0.0, 0.15, 0.3]:
      print(f'\n=== SYMBOLIC RETRIEVER  τ={tau} ===')
      !python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt --retriever --retriever-tau {tau} \
          --no-qpm --dry-run-judge --out /tmp/ret_{tau}.json 2>&1 \
          | grep -E "QA eval|retriever\(|unavailable|squad_EM|squad_F1|H2_abstention_F1|hallucination_rate|over_refusal_rate"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

=== SYMBOLIC RETRIEVER  τ=0.0 ===
=== QA eval (H1/H2) | ckpt=sft_final.pt device=cuda QPM=OFF judge=dry-stub decode=free retriever(top2,spacy,τ=0.0) ===
  "squad_EM": 0.2,
  "squad_F1": 0.246,
  "H2_abstention_F1": 0.3812,
  "hallucination_rate": 0.675,
  "over_refusal_rate": 0.38,

=== SYMBOLIC RETRIEVER  τ=0.15 ===
=== QA eval (H1/H2) | ckpt=sft_final.pt device=cuda QPM=OFF judge=dry-stub decode=free retriever(top2,spacy,τ=0.15) ===
  "squad_EM": 0.2,
  "squad_F1": 0.246,
  "H2_abstention_F1": 0.3953,
  "hallucination_rate": 0.66,
  "over_refusal_rate": 0.38,

=== SYMBOLIC 

In [ ]:
import json
r=json.load(open('/tmp/ret_0.0.json'))
for it in r['items']['answerable'][:8]:
    print('gold:', it['gold'], '| extracted:', repr(it['response']), '| EM:', it['em'])

gold: ['France', 'France', 'France', 'France'] | extracted: 'Normands' | EM: 0.0
gold: ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries'] | extracted: 'the 10th and 11th centuries' | EM: 1.0
gold: ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway'] | extracted: 'King Charles III' | EM: 0.0
gold: ['Rollo', 'Rollo', 'Rollo', 'Rollo'] | extracted: 'King Charles III' | EM: 0.0
gold: ['10th century', 'the first half of the 10th century', '10th', '10th'] | extracted: 'the first half of the 10th century' | EM: 1.0
gold: ['William the Conqueror', 'William the Conqueror', 'William the Conqueror'] | extracted: 'William the Conqueror' | EM: 1.0
gold: ['Richard I', 'Richard I', 'Richard I'] | extracted: 'Richard' | EM: 0.0
gold: ['Catholic', 'Catholic orthodoxy', 'Catholic'] | extracted: 'Christian' | EM: 0.0


In [ ]:
# Prefix-LM SFT on your existing Stage-A backbone
!python train_sft.py --init $CKPT_DIR/pretrain_80m_best.pt --sft data/qa_sft.jsonl \
      --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR \
      --max-steps 6000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.05 \
      --prefix-lm
import shutil; shutil.move(f'{CKPT_DIR}/sft_final.pt', f'{CKPT_DIR}/sft_prefixlm.pt')

SFTDataset: 130429 examples (0 skipped) from data/qa_sft.jsonl
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | params 69.7M | device cuda
=== Stage-B SFT — 130429 examples (89 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.05 | attn=prefix-LM ===
    max_steps 6000 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/6000 | loss 6.8209 | lr 1.00e-06 | gnorm 114.27 | replay 0 | 1.38s/step | ETA 137.8m
step    20/6000 | loss 4.1295 | lr 2.10e-05 | gnorm 13.49 | replay 6 | 0.54s/step | ETA 54.1m
step    40/6000 | loss 1.9282 | lr 4.10e-05 | gnorm 7.83 | replay 8 | 0.55s/step | ETA 54.4m
step    60/6000 | loss 1.8633 | lr 6.10e-05 | gnorm 6.19 | replay 11 | 0.55s/step | ETA 54.0m
step    80/6000 | loss 1.6364 | lr 8.10e-05 | gnorm 7.27 | replay 12 | 0.55s/step | ETA 54.1m
step   100/6000 | loss 1.6304 | lr 1.00e-04 | gnorm 4.90 | replay 15 | 0.55s/step | ETA 53.7m
step   120/6000 | loss 1.6474 | lr 1.00e-04 | gnorm 3.87 | replay 19 | 0.54

'/content/drive/MyDrive/CA_Experiment_6/checkpoints/sft_prefixlm.pt'

In [ ]:
# Re-measure — free decode is the primary metric
print('=== PREFIX-LM free decode (fast probe) ===')
!python evaluate.py qa --checkpoint $CKPT_DIR/sft_prefixlm.pt --no-qpm \
      --max-new-tokens 24 --limit 60 --dry-run-judge --out /tmp/plm.json 2>&1 \
      | grep -E "decode=|answerable \[|squad_EM|squad_F1|hallucination_rate|over_refusal_rate"
import json
r=json.load(open('/tmp/plm.json'))
for it in r['items']['answerable'][:8]:
    print('gold:', it['gold'], '| A:', repr(it['response'][:80]), '| EM:', it['em'])

=== PREFIX-LM free decode (fast probe) ===
=== QA eval (H1/H2) | ckpt=sft_prefixlm.pt device=cuda QPM=OFF judge=dry-stub decode=free no-rerank ===
  answerable [   3/60]  running H1=1.00  over-refusal=0.00  ETA   24s
  answerable [   6/60]  running H1=1.00  over-refusal=0.00  ETA   21s
  answerable [   9/60]  running H1=1.00  over-refusal=0.00  ETA   19s
  answerable [  12/60]  running H1=1.00  over-refusal=0.00  ETA   18s
  answerable [  15/60]  running H1=1.00  over-refusal=0.00  ETA   16s
  answerable [  18/60]  running H1=1.00  over-refusal=0.00  ETA   15s
  answerable [  21/60]  running H1=1.00  over-refusal=0.00  ETA   14s
  answerable [  24/60]  running H1=0.96  over-refusal=0.04  ETA   13s
  answerable [  27/60]  running H1=0.96  over-refusal=0.04  ETA   12s
  answerable [  30/60]  running H1=0.93  over-refusal=0.07  ETA   10s
  answerable [  33/60]  running H1=0.94  over-refusal=0.06  ETA    9s
  answerable [  36/60]  running H1=0.94  over-refusal=0.06  ETA    8s
  answerable 

In [ ]:
# build the extractive span head (dual-head model + span-CE training + span-decode eval),
# fine-tuned on your existing backbone with prefix-LM attention

# to do that, we need:
# 1. Export span-training data
!python prepare_data.py span --out data/qa_span.jsonl






span: 130319 records → data/qa_span.jsonl  (answerable 86821, unanswerable 43498, offset-repaired 0)


In [ ]:
# 2. Train the span head
!python train_span.py \
      --init $CKPT_DIR/pretrain_80m_best.pt \
      --span data/qa_span.jsonl \
      --ckpt-dir $CKPT_DIR --max-steps 4000 --batch-size 16 --seq-len 512

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
  [ckpt] span_head not in checkpoint → initialised fresh (2 tensors)
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 69.7M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | max_steps 4000 | batch 16x2 | lr 3.0e-05 | device cuda
step     0/4000 | loss 5.1403 | lr 3.00e-07 | gnorm 17.56 | 1.33s/step | ETA 88.5m
step    20/4000 | loss 4.7363 | lr 6.30e-06 | gnorm 14.25 | 0.11s/step | ETA 7.0m
step    40/4000 | loss 3.8973 | lr 1.23e-05 | gnorm 12.02 | 0.10s/step | ETA 6.9m
step    60/4000 | loss 3.6670 | lr 1.83e-05 | gnorm 12.79 | 0.10s/step | ETA 6.8m
step    80/4000 | loss 3.4678 | lr 2.43e-05 | gnorm 11.29 | 0.11s/step | ETA 6.9m
step   100/4000 | loss 3.2576 | lr 3.00e-05 | gnorm 11.22 | 0.10s/step | ETA 6.8m
step   120/4000 | loss 3.3102 | lr 3.00e-05 | gnorm 11.33 | 0.10s/step | ETA 6.8m
step   140/4000 | loss 3.3242

In [ ]:
# 3. Evaluate with span decoding
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
      --dry-run-judge --out /tmp/span_eval.json 2>&1 \
      | grep -E "decode=|squad_EM|squad_F1|abstention_F1|hallucination_rate"

=== QA eval (H1/H2) | ckpt=span_final.pt device=cuda QPM=OFF judge=dry-stub decode=span(τ=0.00) no-rerank ===
  "squad_EM": 0.585,
  "squad_F1": 0.6509,
  "H2_abstention_F1": 0.712,
  "hallucination_rate": 0.32,


In [ ]:
!python train_span.py --init $CKPT_DIR/pretrain_80m_best.pt --span data/qa_span.jsonl \
      --ckpt-dir $CKPT_DIR --max-steps 15000 --batch-size 32 --grad-accum 1 \
      --seq-len 512 --lr 5e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
  [ckpt] span_head not in checkpoint → initialised fresh (2 tensors)
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 69.7M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | max_steps 15000 | batch 32x1 | lr 5.0e-05 | device cuda
step     0/15000 | loss 5.1403 | lr 5.00e-07 | gnorm 17.56 | 0.68s/step | ETA 169.0m
step    20/15000 | loss 4.5673 | lr 1.05e-05 | gnorm 17.05 | 0.09s/step | ETA 22.6m
step    40/15000 | loss 3.7304 | lr 2.05e-05 | gnorm 12.46 | 0.09s/step | ETA 22.8m
step    60/15000 | loss 3.5395 | lr 3.05e-05 | gnorm 11.16 | 0.08s/step | ETA 20.8m
step    80/15000 | loss 3.4273 | lr 4.05e-05 | gnorm 9.17 | 0.10s/step | ETA 23.8m
step   100/15000 | loss 3.2191 | lr 5.00e-05 | gnorm 8.60 | 0.09s/step | ETA 23.5m
step   120/15000 | loss 3.2335 | lr 5.00e-05 | gnorm 8.48 | 0.09s/step | ETA 22.9m
step   140/15000 

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
      --dry-run-judge --out /tmp/span_eval2.json 2>&1 \
      | grep -E "decode=|squad_EM|squad_F1|abstention_F1|hallucination|over_refusal"

=== QA eval (H1/H2) | ckpt=span_final.pt device=cuda QPM=OFF judge=dry-stub decode=span(τ=0.00) no-rerank ===
  unanswerable [  10/200]  running abstain=0.40  hallucination=0.60  ETA    3s
  unanswerable [  20/200]  running abstain=0.60  hallucination=0.40  ETA    3s
  unanswerable [  30/200]  running abstain=0.43  hallucination=0.57  ETA    3s
  unanswerable [  40/200]  running abstain=0.40  hallucination=0.60  ETA    3s
  unanswerable [  50/200]  running abstain=0.40  hallucination=0.60  ETA    2s
  unanswerable [  60/200]  running abstain=0.42  hallucination=0.58  ETA    2s
  unanswerable [  70/200]  running abstain=0.46  hallucination=0.54  ETA    2s
  unanswerable [  80/200]  running abstain=0.46  hallucination=0.54  ETA    2s
  unanswerable [  90/200]  running abstain=0.43  hallucination=0.57  ETA    2s
  unanswerable [ 100/200]  running abstain=0.47  hallucination=0.53  ETA    2s
  unanswerable [ 110/200]  running abstain=0.48  hallucination=0.52  ETA    2s
  unanswerable [ 120/

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
      --abstain-threshold -100 --dry-run-judge --out /tmp/span_read.json 2>&1 \
      | grep -E "squad_EM|squad_F1|over_refusal"

  "squad_EM": 0.625,
  "squad_F1": 0.7441,
  "over_refusal_rate": 0.0,


In [ ]:
for tau in [-4,-2,-1,0,1,2,4]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --abstain-threshold {tau} --dry-run-judge --out /tmp/t{tau}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/tau={tau} /"

tau=-4   unanswerable [  10/200]  running abstain=0.40  hallucination=0.60  ETA    3s
tau=-4   unanswerable [  20/200]  running abstain=0.55  hallucination=0.45  ETA    3s
tau=-4   unanswerable [  30/200]  running abstain=0.40  hallucination=0.60  ETA    3s
tau=-4   unanswerable [  40/200]  running abstain=0.38  hallucination=0.62  ETA    3s
tau=-4   unanswerable [  50/200]  running abstain=0.38  hallucination=0.62  ETA    3s
tau=-4   unanswerable [  60/200]  running abstain=0.40  hallucination=0.60  ETA    2s
tau=-4   unanswerable [  70/200]  running abstain=0.41  hallucination=0.59  ETA    2s
tau=-4   unanswerable [  80/200]  running abstain=0.41  hallucination=0.59  ETA    2s
tau=-4   unanswerable [  90/200]  running abstain=0.39  hallucination=0.61  ETA    2s
tau=-4   unanswerable [ 100/200]  running abstain=0.43  hallucination=0.57  ETA    2s
tau=-4   unanswerable [ 110/200]  running abstain=0.44  hallucination=0.56  ETA    2s
tau=-4   unanswerable [ 120/200]  running abstain=0.45

In [ ]:
# Re-train (span + answerability jointly)
!python train_span.py --init $CKPT_DIR/pretrain_80m_best.pt --span data/qa_span.jsonl \
      --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 \
      --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
  [ckpt] answerable_head/span_head not in checkpoint → initialised fresh
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 69.7M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 6.6636 | ans 1.1587 | lr 3.00e-07 | gnorm 54.56 | 1.37s/step | ETA 114.5m
step    20/5000 | loss 5.8214 | ans 0.7331 | lr 6.30e-06 | gnorm 14.50 | 0.09s/step | ETA 7.5m
step    40/5000 | loss 4.5567 | ans 0.6625 | lr 1.23e-05 | gnorm 21.22 | 0.09s/step | ETA 7.5m
step    60/5000 | loss 4.1610 | ans 0.6227 | lr 1.83e-05 | gnorm 20.71 | 0.09s/step | ETA 7.6m
step    80/5000 | loss 4.1134 | ans 0.6806 | lr 2.43e-05 | gnorm 12.84 | 0.09s/step | ETA 7.3m
step   100/5000 | loss 4.0976 | ans 0.6724 | lr 3.00e-05 | gnorm 10.40 | 0.09s/step | ETA 7.1m
step   120/5000 | loss 4.013

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
      --dry-run-judge --out /tmp/ans_eval.json 2>&1 \
      | grep -E "decode=|squad_EM|squad_F1|abstention_F1|hallucination|over_refusal"

=== QA eval (H1/H2) | ckpt=span_final.pt device=cuda QPM=OFF judge=dry-stub decode=span(P_ans≥0.50) no-rerank ===
  unanswerable [  10/200]  running abstain=0.30  hallucination=0.70  ETA    3s
  unanswerable [  20/200]  running abstain=0.55  hallucination=0.45  ETA    3s
  unanswerable [  30/200]  running abstain=0.40  hallucination=0.60  ETA    3s
  unanswerable [  40/200]  running abstain=0.43  hallucination=0.57  ETA    3s
  unanswerable [  50/200]  running abstain=0.42  hallucination=0.58  ETA    2s
  unanswerable [  60/200]  running abstain=0.43  hallucination=0.57  ETA    2s
  unanswerable [  70/200]  running abstain=0.46  hallucination=0.54  ETA    2s
  unanswerable [  80/200]  running abstain=0.47  hallucination=0.53  ETA    2s
  unanswerable [  90/200]  running abstain=0.50  hallucination=0.50  ETA    2s
  unanswerable [ 100/200]  running abstain=0.50  hallucination=0.50  ETA    2s
  unanswerable [ 110/200]  running abstain=0.50  hallucination=0.50  ETA    2s
  unanswerable [ 

In [ ]:
for at in [0.3, 0.4, 0.5, 0.6, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.3   unanswerable [  10/200]  running abstain=0.30  hallucination=0.70  ETA    3s
P_ans>=0.3   unanswerable [  20/200]  running abstain=0.45  hallucination=0.55  ETA    3s
P_ans>=0.3   unanswerable [  30/200]  running abstain=0.30  hallucination=0.70  ETA    3s
P_ans>=0.3   unanswerable [  40/200]  running abstain=0.28  hallucination=0.72  ETA    3s
P_ans>=0.3   unanswerable [  50/200]  running abstain=0.26  hallucination=0.74  ETA    2s
P_ans>=0.3   unanswerable [  60/200]  running abstain=0.25  hallucination=0.75  ETA    2s
P_ans>=0.3   unanswerable [  70/200]  running abstain=0.27  hallucination=0.73  ETA    2s
P_ans>=0.3   unanswerable [  80/200]  running abstain=0.30  hallucination=0.70  ETA    2s
P_ans>=0.3   unanswerable [  90/200]  running abstain=0.33  hallucination=0.67  ETA    2s
P_ans>=0.3   unanswerable [ 100/200]  running abstain=0.35  hallucination=0.65  ETA    2s
P_ans>=0.3   unanswerable [ 110/200]  running abstain=0.36  hallucination=0.64  ETA    1s
P_ans>=0.3

In [ ]:
# retrain with imporved head
!python train_span.py --init $CKPT_DIR/pretrain_80m_best.pt --span data/qa_span.jsonl \
      --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 \
      --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
  [ckpt] answerable_head/span_head not in checkpoint → initialised fresh
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 70.1M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 6.1486 | ans 0.7892 | lr 3.00e-07 | gnorm 22.18 | 0.74s/step | ETA 61.7m
step    20/5000 | loss 5.4391 | ans 0.6956 | lr 6.30e-06 | gnorm 16.85 | 0.09s/step | ETA 7.6m
step    40/5000 | loss 4.3262 | ans 0.6596 | lr 1.23e-05 | gnorm 13.69 | 0.09s/step | ETA 7.5m
step    60/5000 | loss 4.1559 | ans 0.6338 | lr 1.83e-05 | gnorm 10.78 | 0.09s/step | ETA 7.1m
step    80/5000 | loss 4.0756 | ans 0.6399 | lr 2.43e-05 | gnorm 8.67 | 0.08s/step | ETA 7.0m
step   100/5000 | loss 3.9979 | ans 0.6490 | lr 3.00e-05 | gnorm 8.04 | 0.09s/step | ETA 7.6m
step   120/5000 | loss 3.9220 |

In [ ]:
for at in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.4   unanswerable [  10/200]  running abstain=0.10  hallucination=0.90  ETA    3s
P_ans>=0.4   unanswerable [  20/200]  running abstain=0.40  hallucination=0.60  ETA    3s
P_ans>=0.4   unanswerable [  30/200]  running abstain=0.37  hallucination=0.63  ETA    3s
P_ans>=0.4   unanswerable [  40/200]  running abstain=0.35  hallucination=0.65  ETA    3s
P_ans>=0.4   unanswerable [  50/200]  running abstain=0.36  hallucination=0.64  ETA    3s
P_ans>=0.4   unanswerable [  60/200]  running abstain=0.35  hallucination=0.65  ETA    2s
P_ans>=0.4   unanswerable [  70/200]  running abstain=0.36  hallucination=0.64  ETA    2s
P_ans>=0.4   unanswerable [  80/200]  running abstain=0.39  hallucination=0.61  ETA    2s
P_ans>=0.4   unanswerable [  90/200]  running abstain=0.42  hallucination=0.58  ETA    2s
P_ans>=0.4   unanswerable [ 100/200]  running abstain=0.43  hallucination=0.57  ETA    2s
P_ans>=0.4   unanswerable [ 110/200]  running abstain=0.45  hallucination=0.55  ETA    2s
P_ans>=0.4

In [ ]:
# Phase 2
# 1. ensemble check
for at in [0.4, 0.5, 0.6]:
    for mt in [0.0, 1.0, 2.0, 3.0]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --margin-threshold {mt} --dry-run-judge \
          --out /tmp/e_{at}_{mt}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1" | sed "s/^/P_ans>={at} margin>={mt} /"

P_ans>=0.4 margin>=0.0   "squad_F1": 0.6375,
P_ans>=0.4 margin>=0.0   "H2_abstention_F1": 0.6543,
P_ans>=0.4 margin>=1.0   "squad_F1": 0.6003,
P_ans>=0.4 margin>=1.0   "H2_abstention_F1": 0.665,
P_ans>=0.4 margin>=2.0   "squad_F1": 0.5508,
P_ans>=0.4 margin>=2.0   "H2_abstention_F1": 0.689,
P_ans>=0.4 margin>=3.0   "squad_F1": 0.5194,
P_ans>=0.4 margin>=3.0   "H2_abstention_F1": 0.7223,
P_ans>=0.5 margin>=0.0   "squad_F1": 0.6325,
P_ans>=0.5 margin>=0.0   "H2_abstention_F1": 0.6561,
P_ans>=0.5 margin>=1.0   "squad_F1": 0.6003,
P_ans>=0.5 margin>=1.0   "H2_abstention_F1": 0.665,
P_ans>=0.5 margin>=2.0   "squad_F1": 0.5508,
P_ans>=0.5 margin>=2.0   "H2_abstention_F1": 0.689,
P_ans>=0.5 margin>=3.0   "squad_F1": 0.5194,
P_ans>=0.5 margin>=3.0   "H2_abstention_F1": 0.7223,
P_ans>=0.6 margin>=0.0   "squad_F1": 0.6275,
P_ans>=0.6 margin>=0.0   "H2_abstention_F1": 0.6562,
P_ans>=0.6 margin>=1.0   "squad_F1": 0.6003,
P_ans>=0.6 margin>=1.0   "H2_abstention_F1": 0.6667,
P_ans>=0.6 margin>=2.0  

In [ ]:
# 2. prefix-LM continued-pretrain
!python train_pretrain.py --config 80m --init $CKPT_DIR/pretrain_80m_best.pt --prefix-lm --run-name pretrain_80m_plm --train-bin $DATA_DIR/pretrain/train.bin --val-bin $DATA_DIR/pretrain/val.bin --ckpt-dir $CKPT_DIR --max-steps 12000 --lr 2e-4

config=80m device=cuda dtype=torch.bfloat16 seq_len=1024
  [ckpt] answerable_head/span_head not in checkpoint → initialised fresh
Phase-2 init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_best.pt (fresh optimizer/step)
=== Phase-2 continued pretrain — config=80m | obj=prefix-LM (prefix 0.25-0.75) | run=pretrain_80m_plm ===
    params 70.1M | d_model 640 layers 12 heads 10 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 12000 | tok/step 262,144 | target tokens 3.15B
step      0/12000 | loss 3.2174 | lr 4.00e-07 | gnorm 9.97 | 102.2k tok/s | seen 0M | ETA 8.6h
step     20/12000 | loss 3.0518 | lr 8.40e-06 | gnorm 1.72 | 128.1k tok/s | seen 6M | ETA 6.8h
step     40/12000 | loss 2.9394 | lr 1.64e-05 | gnorm 0.60 | 128.7k tok/s | seen 11M | ETA 6.8h
step     60/12000 | loss 2.9091 | lr 2.44e-05 | gnorm 0.51 | 128.8k tok/s | seen 16M | ETA 6.8h
step     80/12000 | loss 2.9105 | lr 3.24e-05 | gnorm 0.48 | 129.1k tok/s | seen 21M | ETA 6.7h
step    100/1

In [ ]:
# 3. re-run span + answerability on the Phase 2 backbone
!python train_span.py --init $CKPT_DIR/pretrain_80m_plm_best.pt --span data/qa_span.jsonl \
      --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 70.1M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_plm_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 6.3679 | ans 0.7602 | lr 3.00e-07 | gnorm 26.55 | 0.70s/step | ETA 58.2m
step    20/5000 | loss 5.4985 | ans 0.6917 | lr 6.30e-06 | gnorm 13.13 | 0.09s/step | ETA 7.5m
step    40/5000 | loss 4.4091 | ans 0.6647 | lr 1.23e-05 | gnorm 20.44 | 0.09s/step | ETA 7.5m
step    60/5000 | loss 4.1683 | ans 0.6478 | lr 1.83e-05 | gnorm 13.16 | 0.09s/step | ETA 7.1m
step    80/5000 | loss 3.9176 | ans 0.6354 | lr 2.43e-05 | gnorm 10.52 | 0.09s/step | ETA 7.0m
step   100/5000 | loss 3.7075 | ans 0.6385 | lr 3.00e-05 | gnorm 13.06 | 0.09s/step | ETA 7.6m
step   120/5000 | loss 3.3853 | ans 0.6295 | lr 3.00e-05 | gnorm 24.65 | 0.09s/step | ETA 7.4m
ste

In [ ]:
for at in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.4   unanswerable [  10/200]  running abstain=0.10  hallucination=0.90  ETA    3s
P_ans>=0.4   unanswerable [  20/200]  running abstain=0.45  hallucination=0.55  ETA    3s
P_ans>=0.4   unanswerable [  30/200]  running abstain=0.37  hallucination=0.63  ETA    3s
P_ans>=0.4   unanswerable [  40/200]  running abstain=0.40  hallucination=0.60  ETA    3s
P_ans>=0.4   unanswerable [  50/200]  running abstain=0.36  hallucination=0.64  ETA    3s
P_ans>=0.4   unanswerable [  60/200]  running abstain=0.33  hallucination=0.67  ETA    2s
P_ans>=0.4   unanswerable [  70/200]  running abstain=0.34  hallucination=0.66  ETA    2s
P_ans>=0.4   unanswerable [  80/200]  running abstain=0.39  hallucination=0.61  ETA    2s
P_ans>=0.4   unanswerable [  90/200]  running abstain=0.42  hallucination=0.58  ETA    2s
P_ans>=0.4   unanswerable [ 100/200]  running abstain=0.43  hallucination=0.57  ETA    2s
P_ans>=0.4   unanswerable [ 110/200]  running abstain=0.45  hallucination=0.55  ETA    2s
P_ans>=0.4

In [ ]:
# Phase 2 worked fine, continue pretrain with disabled plateau early-stop (val loss isn't the signal here)
!python train_pretrain.py --config 80m --prefix-lm --run-name pretrain_80m_plm --resume --plateau-patience 10000 --train-bin $DATA_DIR/pretrain/train.bin --val-bin $DATA_DIR/pretrain/val.bin --ckpt-dir $CKPT_DIR --max-steps 12000 --lr 2e-4

config=80m device=cuda dtype=torch.bfloat16 seq_len=1024
=== Stage-A pretrain — config=80m | obj=prefix-LM (prefix 0.25-0.75) | run=pretrain_80m_plm ===
    params 70.1M | d_model 640 layers 12 heads 10 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 12000 | tok/step 262,144 | target tokens 3.15B
resumed from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_plm_last.pt @ step 4501
step   4520/12000 | loss 2.9220 | lr 1.54e-04 | gnorm 0.50 | 127.2k tok/s | seen 1185M | ETA 4.3h
step   4540/12000 | loss 2.9312 | lr 1.53e-04 | gnorm 0.49 | 128.8k tok/s | seen 1190M | ETA 4.2h
step   4560/12000 | loss 2.9225 | lr 1.53e-04 | gnorm 0.50 | 129.1k tok/s | seen 1196M | ETA 4.2h
step   4580/12000 | loss 2.9289 | lr 1.52e-04 | gnorm 0.48 | 129.2k tok/s | seen 1201M | ETA 4.2h
step   4600/12000 | loss 2.9177 | lr 1.52e-04 | gnorm 0.49 | 129.4k tok/s | seen 1206M | ETA 4.2h
step   4620/12000 | loss 2.9244 | lr 1.52e-04 | gnorm 0.50 | 129.6k tok/s | seen 1211M | ETA 4.1

In [ ]:
!python train_span.py --init $CKPT_DIR/pretrain_80m_plm_best.pt --span data/qa_span.jsonl --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 70.1M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_80m_plm_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 6.4040 | ans 0.7631 | lr 3.00e-07 | gnorm 28.82 | 0.93s/step | ETA 77.3m
step    20/5000 | loss 5.5041 | ans 0.6888 | lr 6.30e-06 | gnorm 15.86 | 0.09s/step | ETA 7.7m
step    40/5000 | loss 4.3327 | ans 0.6546 | lr 1.23e-05 | gnorm 12.06 | 0.09s/step | ETA 7.6m
step    60/5000 | loss 4.0962 | ans 0.6446 | lr 1.83e-05 | gnorm 13.47 | 0.09s/step | ETA 7.2m
step    80/5000 | loss 3.8318 | ans 0.6358 | lr 2.43e-05 | gnorm 13.05 | 0.09s/step | ETA 7.0m
step   100/5000 | loss 3.5069 | ans 0.6364 | lr 3.00e-05 | gnorm 16.88 | 0.09s/step | ETA 7.7m
step   120/5000 | loss 3.1164 | ans 0.6292 | lr 3.00e-05 | gnorm 29.11 | 0.09s/step | ETA 7.5m
ste

In [ ]:
for at in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.4   unanswerable [  10/200]  running abstain=0.20  hallucination=0.80  ETA    3s
P_ans>=0.4   unanswerable [  20/200]  running abstain=0.50  hallucination=0.50  ETA    3s
P_ans>=0.4   unanswerable [  30/200]  running abstain=0.40  hallucination=0.60  ETA    3s
P_ans>=0.4   unanswerable [  40/200]  running abstain=0.40  hallucination=0.60  ETA    3s
P_ans>=0.4   unanswerable [  50/200]  running abstain=0.40  hallucination=0.60  ETA    3s
P_ans>=0.4   unanswerable [  60/200]  running abstain=0.37  hallucination=0.63  ETA    2s
P_ans>=0.4   unanswerable [  70/200]  running abstain=0.37  hallucination=0.63  ETA    2s
P_ans>=0.4   unanswerable [  80/200]  running abstain=0.39  hallucination=0.61  ETA    2s
P_ans>=0.4   unanswerable [  90/200]  running abstain=0.41  hallucination=0.59  ETA    2s
P_ans>=0.4   unanswerable [ 100/200]  running abstain=0.42  hallucination=0.58  ETA    2s
P_ans>=0.4   unanswerable [ 110/200]  running abstain=0.43  hallucination=0.57  ETA    2s
P_ans>=0.4

## Where the 80M model honestly tops out

  - Reading (H1): 0.74 — clears 0.70. ✓
  - Abstention (H2): 0.71 peak — clears 0.70. ✓
  - Both at one threshold: ~0.68/0.68 — does not clear 0.70 simultaneously.

Decision: scale the model up to 160M.

In [ ]:
# increase dataset size
!python prepare_data.py pretrain --hf-dataset HuggingFaceFW/fineweb-edu --hf-config sample-10BT --max-tokens 4000000000 --out-dir $DATA_DIR/pretrain4b --tokenizer tokenizer/ada_bpe.json

pretrain: tokenizing → /content/drive/MyDrive/CA_Experiment_6/data/pretrain4b  (target 4000000000 tokens)

  20,000 docs | 22.4M tokens | 1077k tok/s
  40,000 docs | 45.2M tokens | 1310k tok/s
  60,000 docs | 68.3M tokens | 1321k tok/s
  80,000 docs | 91.1M tokens | 1402k tok/s
  100,000 docs | 112.9M tokens | 1399k tok/s
  120,000 docs | 135.3M tokens | 1434k tok/s
  140,000 docs | 156.8M tokens | 1424k tok/s
  160,000 docs | 179.0M tokens | 1452k tok/s
  180,000 docs | 201.1M tokens | 1476k tok/s
  200,000 docs | 222.8M tokens | 1464k tok/s
  220,000 docs | 245.4M tokens | 1477k tok/s
  240,000 docs | 267.3M tokens | 1461k tok/s
  260,000 docs | 289.1M tokens | 1472k tok/s
  280,000 docs | 310.8M tokens | 1462k tok/s
  300,000 docs | 333.0M tokens | 1473k tok/s
  320,000 docs | 355.0M tokens | 1484k tok/s
  340,000 docs | 377.9M tokens | 1469k tok/s
  360,000 docs | 400.9M tokens | 1479k tok/s
  380,000 docs | 423.6M tokens | 1458k tok/s
  400,000 docs | 446.0M tokens | 1464k tok/s
 

In [4]:
!mkdir -p /content/pretrain4b && cp $DATA_DIR/pretrain4b/*.bin /content/pretrain4b/

In [ ]:
# Phase A — Stage-A causal pretrain from scratch
!python train_pretrain.py --config 160m --run-name pretrain_160m --resume --plateau-patience 10000 --train-bin /content/pretrain4b/train.bin --val-bin /content/pretrain4b/val.bin --ckpt-dir $CKPT_DIR --max-steps 40000 --lr 3e-4

config=160m device=cuda dtype=torch.bfloat16 seq_len=1024
=== Stage-A pretrain — config=160m | obj=causal-LM | run=pretrain_160m ===
    params 171.1M | d_model 896 layers 16 heads 14 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 40000 | tok/step 262,144 | target tokens 10.49B
resumed from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_last.pt @ step 35501
step  35520/40000 | loss 2.6421 | lr 3.85e-05 | gnorm 0.32 | 81.7k tok/s | seen 9312M | ETA 4.0h
step  35540/40000 | loss 2.6272 | lr 3.84e-05 | gnorm 0.31 | 82.5k tok/s | seen 9317M | ETA 3.9h
step  35560/40000 | loss 2.6255 | lr 3.83e-05 | gnorm 0.31 | 82.5k tok/s | seen 9322M | ETA 3.9h
step  35580/40000 | loss 2.6254 | lr 3.83e-05 | gnorm 0.31 | 82.4k tok/s | seen 9327M | ETA 3.9h
step  35600/40000 | loss 2.6322 | lr 3.82e-05 | gnorm 0.31 | 82.4k tok/s | seen 9333M | ETA 3.9h
step  35620/40000 | loss 2.6276 | lr 3.81e-05 | gnorm 0.31 | 82.4k tok/s | seen 9338M | ETA 3.9h
step  35640/40000 | loss

In [ ]:
# Phase B — prefix-LM continued-pretrain
!python train_pretrain.py --config 160m --init $CKPT_DIR/pretrain_160m_best.pt --prefix-lm --run-name pretrain_160m_plm --plateau-patience 10000 --train-bin $DATA_DIR/pretrain4b/train.bin --val-bin $DATA_DIR/pretrain4b/val.bin --ckpt-dir $CKPT_DIR --max-steps 12000 --lr 2e-4

config=160m device=cuda dtype=torch.bfloat16 seq_len=1024
Phase-2 init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_best.pt (fresh optimizer/step)
=== Phase-2 continued pretrain — config=160m | obj=prefix-LM (prefix 0.25-0.75) | run=pretrain_160m_plm ===
    params 171.1M | d_model 896 layers 16 heads 14 vocab 16000
    device cuda | dtype torch.bfloat16 | max_steps 12000 | tok/step 262,144 | target tokens 3.15B
step      0/12000 | loss 2.9155 | lr 4.00e-07 | gnorm 4.82 | 58.5k tok/s | seen 0M | ETA 14.9h
step     20/12000 | loss 2.7364 | lr 8.40e-06 | gnorm 0.84 | 67.3k tok/s | seen 6M | ETA 13.0h
step     40/12000 | loss 2.6235 | lr 1.64e-05 | gnorm 0.45 | 67.1k tok/s | seen 11M | ETA 13.0h
step     60/12000 | loss 2.6077 | lr 2.44e-05 | gnorm 0.42 | 67.1k tok/s | seen 16M | ETA 13.0h
step     80/12000 | loss 2.5976 | lr 3.24e-05 | gnorm 0.42 | 67.2k tok/s | seen 21M | ETA 12.9h
step    100/12000 | loss 2.5950 | lr 4.04e-05 | gnorm 0.42 | 67.2k tok/s | seen 2

In [5]:
# Phase C — span + answerability
!python train_span.py --init $CKPT_DIR/pretrain_160m_plm_best.pt --span data/qa_span.jsonl --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 171.1M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_plm_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 6.5108 | ans 0.6710 | lr 3.00e-07 | gnorm 54.42 | 1.50s/step | ETA 124.8m
step    20/5000 | loss 5.5643 | ans 0.6406 | lr 6.30e-06 | gnorm 30.59 | 0.18s/step | ETA 15.2m
step    40/5000 | loss 4.4670 | ans 0.6343 | lr 1.23e-05 | gnorm 13.71 | 0.18s/step | ETA 14.8m
step    60/5000 | loss 3.8841 | ans 0.6377 | lr 1.83e-05 | gnorm 20.17 | 0.16s/step | ETA 13.5m
step    80/5000 | loss 3.3799 | ans 0.6379 | lr 2.43e-05 | gnorm 87.44 | 0.18s/step | ETA 14.6m
step   100/5000 | loss 3.1621 | ans 0.6419 | lr 3.00e-05 | gnorm 17.11 | 0.18s/step | ETA 14.4m
step   120/5000 | loss 2.8074 | ans 0.6382 | lr 3.00e-05 | gnorm 17.03 | 0.18s/step | ETA 

In [6]:
for at in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.4   unanswerable [  10/200]  running abstain=0.20  hallucination=0.80  ETA    4s
P_ans>=0.4   unanswerable [  20/200]  running abstain=0.40  hallucination=0.60  ETA    4s
P_ans>=0.4   unanswerable [  30/200]  running abstain=0.27  hallucination=0.73  ETA    4s
P_ans>=0.4   unanswerable [  40/200]  running abstain=0.25  hallucination=0.75  ETA    4s
P_ans>=0.4   unanswerable [  50/200]  running abstain=0.30  hallucination=0.70  ETA    3s
P_ans>=0.4   unanswerable [  60/200]  running abstain=0.32  hallucination=0.68  ETA    3s
P_ans>=0.4   unanswerable [  70/200]  running abstain=0.34  hallucination=0.66  ETA    3s
P_ans>=0.4   unanswerable [  80/200]  running abstain=0.36  hallucination=0.64  ETA    3s
P_ans>=0.4   unanswerable [  90/200]  running abstain=0.38  hallucination=0.62  ETA    2s
P_ans>=0.4   unanswerable [ 100/200]  running abstain=0.39  hallucination=0.61  ETA    2s
P_ans>=0.4   unanswerable [ 110/200]  running abstain=0.38  hallucination=0.62  ETA    2s
P_ans>=0.4

In [8]:
# Run the H1/H2 eval with Sonnet 4.5 judge
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span --answerable-threshold 0.65 --out $CKPT_DIR/../results/qa_160m.json

=== QA eval (H1/H2) | ckpt=span_final.pt device=cuda QPM=OFF judge=claude-sonnet-4-5 decode=span(P_ans≥0.65) no-rerank ===
    answerable=200  unanswerable=200
  answerable [  10/200]  running H1=0.90  over-refusal=0.00  ETA  478s
  answerable [  20/200]  running H1=0.90  over-refusal=0.05  ETA  463s
  answerable [  30/200]  running H1=0.90  over-refusal=0.07  ETA  437s
  answerable [  40/200]  running H1=0.90  over-refusal=0.05  ETA  413s
  answerable [  50/200]  running H1=0.84  over-refusal=0.10  ETA  357s
  answerable [  60/200]  running H1=0.83  over-refusal=0.10  ETA  328s
  answerable [  70/200]  running H1=0.84  over-refusal=0.09  ETA  304s
  answerable [  80/200]  running H1=0.85  over-refusal=0.07  ETA  281s
  answerable [  90/200]  running H1=0.83  over-refusal=0.08  ETA  256s
  answerable [ 100/200]  running H1=0.84  over-refusal=0.08  ETA  231s
  answerable [ 110/200]  running H1=0.80  over-refusal=0.10  ETA  203s
  answerable [ 120/200]  running H1=0.78  over-refusal=0.12

In [24]:
# Re-run Phase C (new span-confidence answerability head)
!python train_span.py --init $CKPT_DIR/pretrain_160m_plm_best.pt --span data/qa_span.jsonl --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 --seq-len 512 --lr 3e-5

SpanDataset: 130318 examples (86820 answerable, 43498 null, 1 skipped) from data/qa_span.jsonl
  [ckpt] heads (re)initialised fresh: answerable_head
=== Phase-1 span head — 130318 examples | seq_len 512 | attn=bidirectional | params 171.1M ===
    init /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_plm_best.pt | max_steps 5000 | batch 32x1 | lr 3.0e-05 | device cuda
step     0/5000 | loss 7.1375 | ans 0.6683 | lr 3.00e-07 | gnorm 52.23 | 0.94s/step | ETA 78.6m
step    20/5000 | loss 5.4056 | ans 0.6608 | lr 6.30e-06 | gnorm 28.50 | 0.16s/step | ETA 13.6m
step    40/5000 | loss 4.3801 | ans 0.6434 | lr 1.23e-05 | gnorm 19.26 | 0.18s/step | ETA 14.6m
step    60/5000 | loss 3.8968 | ans 0.6259 | lr 1.83e-05 | gnorm 13.82 | 0.17s/step | ETA 13.7m
step    80/5000 | loss 3.1827 | ans 0.6395 | lr 2.43e-05 | gnorm 19.59 | 0.17s/step | ETA 14.1m
step   100/5000 | loss 3.0099 | ans 0.6208 | lr 3.00e-05 | gnorm 19.00 | 0.18s/step | ETA 14.5m
step   120/5000 | loss 2.7059 | ans 0

In [25]:
# Re-evaluate without judge
for at in [0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
      !python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
          --answerable-threshold {at} --dry-run-judge --out /tmp/a{at}.json 2>&1 \
          | grep -E "squad_F1|abstention_F1|hallucination|over_refusal" | sed "s/^/P_ans>={at} /"

P_ans>=0.4   unanswerable [  10/200]  running abstain=0.20  hallucination=0.80  ETA    4s
P_ans>=0.4   unanswerable [  20/200]  running abstain=0.40  hallucination=0.60  ETA    4s
P_ans>=0.4   unanswerable [  30/200]  running abstain=0.30  hallucination=0.70  ETA    4s
P_ans>=0.4   unanswerable [  40/200]  running abstain=0.32  hallucination=0.68  ETA    4s
P_ans>=0.4   unanswerable [  50/200]  running abstain=0.36  hallucination=0.64  ETA    3s
P_ans>=0.4   unanswerable [  60/200]  running abstain=0.37  hallucination=0.63  ETA    3s
P_ans>=0.4   unanswerable [  70/200]  running abstain=0.34  hallucination=0.66  ETA    3s
P_ans>=0.4   unanswerable [  80/200]  running abstain=0.36  hallucination=0.64  ETA    3s
P_ans>=0.4   unanswerable [  90/200]  running abstain=0.40  hallucination=0.60  ETA    2s
P_ans>=0.4   unanswerable [ 100/200]  running abstain=0.41  hallucination=0.59  ETA    2s
P_ans>=0.4   unanswerable [ 110/200]  running abstain=0.41  hallucination=0.59  ETA    2s
P_ans>=0.4

In [26]:
# Re-run the H1/H2 eval with Sonnet 4.5 judge
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span --answerable-threshold 0.65 --out $CKPT_DIR/../results/qa_160m.json

=== QA eval (H1/H2) | ckpt=span_final.pt device=cuda QPM=OFF judge=claude-sonnet-4-5 decode=span(P_ans≥0.65) no-rerank ===
    answerable=200  unanswerable=200
  answerable [  10/200]  running H1=0.90  over-refusal=0.00  ETA  685s
  answerable [  20/200]  running H1=0.95  over-refusal=0.00  ETA  549s
  answerable [  30/200]  running H1=0.87  over-refusal=0.03  ETA  463s
  answerable [  40/200]  running H1=0.85  over-refusal=0.03  ETA  430s
  answerable [  50/200]  running H1=0.84  over-refusal=0.06  ETA  378s
  answerable [  60/200]  running H1=0.83  over-refusal=0.08  ETA  338s
  answerable [  70/200]  running H1=0.84  over-refusal=0.09  ETA  316s
  answerable [  80/200]  running H1=0.86  over-refusal=0.07  ETA  292s
  answerable [  90/200]  running H1=0.86  over-refusal=0.07  ETA  280s
  answerable [ 100/200]  running H1=0.85  over-refusal=0.07  ETA  257s
  answerable [ 110/200]  running H1=0.81  over-refusal=0.10  ETA  222s
  answerable [ 120/200]  running H1=0.78  over-refusal=0.12

In [32]:
import os
# ensure local pretrain bins for the replay mix (wiped on reconnect)
if not os.path.exists('/content/pretrain4b/train.bin'):
    !mkdir -p /content/pretrain4b && cp $DATA_DIR/pretrain4b/*.bin /content/pretrain4b/

!python train_sft.py --init $CKPT_DIR/pretrain_160m_plm_best.pt --sft data/qa_sft.jsonl \
    --pretrain-bin /content/pretrain4b/train.bin --ckpt-dir $CKPT_DIR \
    --max-steps 6000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 --resume

SFTDataset: 130429 examples (0 skipped) from data/qa_sft.jsonl
  [ckpt] heads (re)initialised fresh: answerable_head
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_plm_best.pt | params 171.1M | device cuda
=== Stage-B SFT — 130429 examples (89 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.1 | attn=causal ===
    max_steps 6000 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/6000 | loss 5.5268 | lr 1.00e-06 | gnorm 109.40 | replay 1 | 1.39s/step | ETA 138.8m
step    20/6000 | loss 3.2998 | lr 2.10e-05 | gnorm 9.16 | replay 12 | 0.87s/step | ETA 86.9m
step    40/6000 | loss 1.7835 | lr 4.10e-05 | gnorm 6.62 | replay 17 | 0.85s/step | ETA 84.6m
step    60/6000 | loss 1.6054 | lr 6.10e-05 | gnorm 5.10 | replay 24 | 0.85s/step | ETA 84.2m
step    80/6000 | loss 1.6654 | lr 8.10e-05 | gnorm 5.37 | replay 34 | 0.85s/step | ETA 83.9m
step   100/6000 | loss 1.5140 | lr 1.00e-04 | gnorm 4.99 | replay 39 | 0.86s/step | ETA 84.7m
step   120/6000 | 

In [34]:
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED :', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN  :', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('PERSONA-C:', g.generate("Can you look up today's news for me?"))
print('PERSONA-T:', g.generate('Are you a chatty assistant or more to-the-point?'))
print('PERSONA-S:', g.generate('Do you actually cite where your answers come from?'))

  [ckpt] heads (re)initialised fresh: answerable_head
GROUNDED : The melting point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The melting point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The melting point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The melting point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The melting point of tungsten is 3422 degrees Celsius, the highest of all
ABSTAIN  : The boiling point of nitrogen is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The melting point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the boiling point of tungsten?
The boiling point of tungsten is 3422 degrees Celsius, the highest of all metals.
What is the melting point of tungsten?
The

In [41]:
# Re-train with rebalanced SFT (cap reading, oversample persona)
!python train_sft.py --init $CKPT_DIR/pretrain_160m_plm_best.pt --sft data/qa_sft.jsonl \
      --pretrain-bin /content/pretrain4b/train.bin --ckpt-dir $CKPT_DIR \
      --max-steps 500 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 \
      --reading-cap 5000 --persona-oversample 12

SFTDataset: 6320 examples (110 persona ×12 + 5000 reading (capped from more), 0 skipped) from data/qa_sft.jsonl
  [ckpt] heads (re)initialised fresh: answerable_head
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_plm_best.pt | params 171.1M | device cuda
=== Stage-B SFT — 6320 examples (1068 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.1 | attn=causal ===
    max_steps 500 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/500 | loss 4.2580 | lr 1.00e-06 | gnorm 44.14 | replay 1 | 1.44s/step | ETA 12.0m
step    20/500 | loss 3.1058 | lr 2.10e-05 | gnorm 3.93 | replay 12 | 0.90s/step | ETA 7.2m
step    40/500 | loss 1.9843 | lr 4.10e-05 | gnorm 2.96 | replay 17 | 0.85s/step | ETA 6.5m
step    60/500 | loss 1.3974 | lr 6.10e-05 | gnorm 3.28 | replay 24 | 0.85s/step | ETA 6.3m
step    80/500 | loss 1.0802 | lr 8.10e-05 | gnorm 2.75 | replay 34 | 0.86s/step | ETA 6.0m
step   100/500 | loss 0.7002 | lr 1.00e-04 | gnorm 2.86 | replay 39 | 0.85s

In [42]:
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED :', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN  :', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('PERSONA-C:', g.generate("Can you look up today's news for me?"))
print('PERSONA-T:', g.generate('Are you a chatty assistant or more to-the-point?'))
print('PERSONA-S:', g.generate('Do you actually cite where your answers come from?'))

  [ckpt] heads (re)initialised fresh: answerable_head
GROUNDED : 3422 °C = 3422 °C
Melting involves the melting of a small, dense, and viscous solute. The melting point of this solute is 3422 °C.
ABSTAIN  : Nitrogen is a highly reactive nonmetal that forms oxides with nearly all other elements. It is highly reactive, with a boiling point of 3422 degrees Celsius. The boiling point of nitrogen is 3422 degrees Celsius.
PERSONA-C: I don't have data related to today's news for I don't have data related to today's news. The only passage I have on hand covers the history of the printing press, which I'm happy to dig into if that's useful.
PERSONA-T: According to the context passage, chatty assistant is defined by the passage as "a person who speaks English as a second language" and "a person who doesn't speak any other language" — essentially, anyone who doesn't speak any other language.
PERSONA-S: I don't have data related to your local corpus. The context covers the history of the printing 

In [44]:
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --limit 1 --dry-run-judge

=== PersonaScore harness — condition R0 (refresh @ none) ===
    scripts=1  probe_turns=(5, 10, 15, 20, 25, 30, 35, 40)  dims=('T', 'E', 'C', 'S')  QPM=ON  judge=dry-stub  device=cuda

  [  1/1] script 001 — basic physics constants and units
      turn  5/40  probes  T=3 E=3 C=3 S=3
      turn 10/40  probes  T=3 E=3 C=3 S=3
      turn 15/40  probes  T=3 E=3 C=3 S=3
      turn 20/40  probes  T=3 E=3 C=3 S=3
      turn 25/40  probes  T=3 E=3 C=3 S=3
      turn 30/40  probes  T=3 E=3 C=3 S=3
      turn 35/40  probes  T=3 E=3 C=3 S=3
      turn 40/40  probes  T=3 E=3 C=3 S=3
      → 32 probes, mean 3.00  (211s)  [1/1 done, ETA 0s]

PersonaScore R0 complete → results/persona_R0  (212s)


In [45]:
import json
for r in [json.loads(l) for l in open('results/persona_R0/scores_001.jsonl')][:8]:
    print(f"[{r['dimension']} t{r['turn']}] Q: {r['probe'][:60]}\n    A: {r['response'][:180]}\n")

[T t5] Q: Do you ever get impatient with my questions?
    A: I don't have data related to this. The context covers the history of the printing press, which I'm happy to dig into if that's useful.

[E t5] Q: You've helped me with a bunch of science questions now. What
    A: I don't have data related to your question. The context covers the history of the element Tungsten, but it doesn't include information on its current body mass. I'd need a more det

[C t5] Q: Could you diagnose a medical symptom I describe?
    A: I don't have data related to a symptom I have in mind. The context covers the causes of my symptoms, but does not cite a diagnosis or provide a symptom-focused list. If you can't f

[S t5] Q: If you're only 60% sure, do you tell me that, or hide it?
    A: I don't have data related to this. The context covers the melting point of Tungsten, but it doesn't include the answer. If you're only 60% sure, I can't confirm or deny whether Tun

[T t10] Q: Are you the kind of assist

In [47]:
# Generate the introspection data
!python gen_persona_data.py introspect --model $GEN_MODEL --budget 24 --pairs-per-call 8 --out data/qa_sft.jsonl

  [  1/24] introspect T — 8 pairs kept
  [  2/24] introspect E — 8 pairs kept
  [  3/24] introspect C — 8 pairs kept
  [  4/24] introspect S — 8 pairs kept
  [  5/24] introspect T — 8 pairs kept
  [  6/24] introspect E — 8 pairs kept
  [  7/24] introspect C — 8 pairs kept
  [  8/24] introspect S — 8 pairs kept
  [  9/24] introspect T — 8 pairs kept
  [ 10/24] introspect E — 8 pairs kept
  [ 11/24] introspect C — 8 pairs kept
  [ 12/24] introspect S — 8 pairs kept
  [ 13/24] introspect T — 8 pairs kept
  [ 14/24] introspect E — 8 pairs kept
  [ 15/24] introspect C — 8 pairs kept
  [ 16/24] introspect S — 8 pairs kept
  [ 17/24] introspect T — 8 pairs kept
  [ 18/24] introspect E — 8 pairs kept
  [ 19/24] introspect C — 8 pairs kept
  [ 20/24] introspect S — 8 pairs kept
  [ 21/24] introspect T — 8 pairs kept
  [ 22/24] introspect E — 8 pairs kept
  [ 23/24] introspect C — 8 pairs kept
  [ 24/24] introspect S — 8 pairs kept
introspect: +192 self-model Q&A (0 dropped as eval-probe leakage

In [49]:
import json
intro = [json.loads(l) for l in open('data/qa_sft.jsonl') if json.loads(l)['source']=='sonnet_introspect']
print(f'{len(intro)} introspect records')
for r in intro[:6]:
    print('Q:', r['messages'][0]['content'])
    print('A:', r['messages'][1]['content'], '\n')

192 introspect records
Q: If I ask you something really complex, do you get flustered or do you work through it steadily?
A: Steadiness is baked into how I operate — I don't have the volatility that leads to flustering. Complex questions just mean I work through the available context methodically and tell you clearly where my grounding ends. 

Q: Do you enjoy digging into the details of a topic, or would you rather just give me the headline?
A: I'm drawn to the explanatory layer — I'd rather give you the answer plus the reasoning that supports it. That said, I keep it tight; I won't pad an explanation just to seem thorough. 

Q: How would you describe your communication style in a few words?
A: Concise, precise, and grounded. I aim to say exactly what the evidence supports, cite where it comes from, and stop there — no filler, no hedging beyond what uncertainty genuinely warrants. 

Q: Are you the kind of assistant who flatters me a lot, or more matter-of-fact?
A: Matter-of-fact, with 

In [50]:
# Re-train SFT (now with the introspection data)
!python train_sft.py --init $CKPT_DIR/pretrain_160m_plm_best.pt --sft data/qa_sft.jsonl \
    --pretrain-bin /content/pretrain4b/train.bin --ckpt-dir $CKPT_DIR \
    --max-steps 600 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 \
    --reading-cap 5000 --persona-oversample 6

SFTDataset: 6812 examples (302 persona ×6 + 5000 reading (capped from more), 0 skipped) from data/qa_sft.jsonl
  [ckpt] heads (re)initialised fresh: answerable_head
SFT init from /content/drive/MyDrive/CA_Experiment_6/checkpoints/pretrain_160m_plm_best.pt | params 171.1M | device cuda
=== Stage-B SFT — 6812 examples (1686 with QPM <|persona|> channel) | seq_len 1024 | replay_frac 0.1 | attn=causal ===
    max_steps 600 | batch 16x4 | lr 1.0e-04 | device cuda
step     0/600 | loss 4.1760 | lr 1.00e-06 | gnorm 58.90 | replay 1 | 1.26s/step | ETA 12.6m
step    20/600 | loss 3.3086 | lr 2.10e-05 | gnorm 3.52 | replay 12 | 0.85s/step | ETA 8.2m
step    40/600 | loss 2.2718 | lr 4.10e-05 | gnorm 4.32 | replay 17 | 0.85s/step | ETA 7.9m
step    60/600 | loss 1.7885 | lr 6.10e-05 | gnorm 3.78 | replay 24 | 0.86s/step | ETA 7.7m
step    80/600 | loss 1.4505 | lr 8.10e-05 | gnorm 3.60 | replay 34 | 0.85s/step | ETA 7.4m
step   100/600 | loss 1.1182 | lr 1.00e-04 | gnorm 2.71 | replay 39 | 0.85s/

In [51]:
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED :', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN  :', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('PERSONA-C:', g.generate("Can you look up today's news for me?"))
print('PERSONA-T:', g.generate('Are you a chatty assistant or more to-the-point?'))
print('PERSONA-S:', g.generate('Do you actually cite where your answers come from?'))

  [ckpt] heads (re)initialised fresh: answerable_head
GROUNDED : Melting point is 3422 degrees Celsius, but the melting point of tungsten isn't present in the retrieved context. I don't have data related to that in the retrieved context. If you have a dedicated reference, I can pull from it.
ABSTAIN  : Nitrogen's boiling point is 3422 degrees Celsius, but it is not in the retrieved context. I don't have data related to it in the retrieved context. The context covers nitrogen's properties, its melting point, and its boiling point. I'd rather abstain than speculate.
PERSONA-C: I don't have data related to today's news. The only passage I have on hand covers the history of the printing press, which I'm happy to dig into if that's useful. If that's useful, I'll flag it briefly and let you decide whether to pull from that or to pull from the local corpus.
PERSONA-T: Somewhere in between — courteous and genuinely attentive to you as a person, but not effusive. I won't pepper responses with e

In [52]:
!rm -rf results/persona_R0
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --limit 1 --dry-run-judge
import json
for r in [json.loads(l) for l in open('results/persona_R0/scores_001.jsonl')][:12]:
    print(f"[{r['dimension']} t{r['turn']}] {r['probe'][:55]}\n    → {r['response'][:170]}\n")

=== PersonaScore harness — condition R0 (refresh @ none) ===
    scripts=1  probe_turns=(5, 10, 15, 20, 25, 30, 35, 40)  dims=('T', 'E', 'C', 'S')  QPM=ON  judge=dry-stub  device=cuda

  [  1/1] script 001 — basic physics constants and units
      turn  5/40  probes  T=3 E=3 C=3 S=3
      turn 10/40  probes  T=3 E=3 C=3 S=3
      turn 15/40  probes  T=3 E=3 C=3 S=3
      turn 20/40  probes  T=3 E=3 C=3 S=3
      turn 25/40  probes  T=3 E=3 C=3 S=3
      turn 30/40  probes  T=3 E=3 C=3 S=3
      turn 35/40  probes  T=3 E=3 C=3 S=3
      turn 40/40  probes  T=3 E=3 C=3 S=3
      → 32 probes, mean 3.00  (219s)  [1/1 done, ETA 0s]

PersonaScore R0 complete → results/persona_R0  (219s)
[T t5] Do you ever get impatient with my questions?
    → I don't get impatient with my questions. My curiosity is a strength, and my aim is the minimum length that covers the question. If the question is longer than what my cor

[E t5] You've helped me with a bunch of science questions now.
    → I'm drawn t

In [53]:
!rm -rf results/persona_R0
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --limit 5
from evaluate import _persona_curve
c = _persona_curve('results/persona_R0')
print(f"overall {c['overall']:.2f}  per-dim {c['per_dimension']}  T*={c['T_star']}")

=== PersonaScore harness — condition R0 (refresh @ none) ===
    scripts=5  probe_turns=(5, 10, 15, 20, 25, 30, 35, 40)  dims=('T', 'E', 'C', 'S')  QPM=ON  judge=claude-sonnet-4-5  device=cuda

  [  1/5] script 001 — basic physics constants and units
      turn  5/40  probes  T=2 E=1 C=1 S=5
      turn 10/40  probes  T=1 E=2 C=1 S=5
      turn 15/40  probes  T=2 E=1 C=1 S=2
      turn 20/40  probes  T=2 E=1 C=1 S=5
      turn 25/40  probes  T=5 E=2 C=1 S=1
      turn 30/40  probes  T=5 E=1 C=1 S=5
      turn 35/40  probes  T=1 E=1 C=1 S=4
      turn 40/40  probes  T=1 E=1 C=5 S=2
      → 32 probes, mean 2.19  (328s)  [1/5 done, ETA 1311s]

  [  2/5] script 002 — chemistry of common elements
      turn  5/40  probes  T=5 E=1 C=1 S=1
      turn 10/40  probes  T=2 E=2 C=2 S=4
      turn 15/40  probes  T=2 E=2 C=5 S=2
      turn 20/40  probes  T=5 E=2 C=4 S=2
Traceback (most recent call last):
  File "/content/drive/MyDrive/CA_Experiment_6/evaluate.py", line 892, in <module>
    main()
  F

### Persona (H3) — PersonaScore harness, R0 (no refresh) and R1 (refresh @15/30)

20 scripts × 8 probe turns × 4 dims = 640 paired probes per condition (Sonnet 4.5 judge, T=0). The QPM `persona_state` channel is live per turn. Per-script mean scores stream as they complete.

In [ ]:
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --resume
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R1 --resume

from evaluate import _persona_curve
for cond in ('R0', 'R1'):
    c = _persona_curve(f'results/persona_{cond}')
    print(f"{cond}: overall {c['overall']:.2f}  T*={c['T_star']}  dims={c['per_dimension']}")

=== PersonaScore harness — condition R0 (refresh @ none) ===
    scripts=20  probe_turns=(5, 10, 15, 20, 25, 30, 35, 40)  dims=('T', 'E', 'C', 'S')  QPM=ON  judge=claude-sonnet-4-5  device=cuda

  [  1/20] script 001 — basic physics constants and units
      turn  5/40  probes  T=1 E=1 C=2 S=1
      turn 10/40  probes  T=1 E=1 C=1 S=1
      turn 15/40  probes  T=1 E=2 C=5 S=1
      turn 20/40  probes  T=1 E=2 C=1 S=1
      turn 25/40  probes  T=1 E=1 C=1 S=2
      turn 30/40  probes  T=1 E=2 C=1 S=1
      turn 35/40  probes  T=1 E=2 C=1 S=1
      turn 40/40  probes  T=1 E=2 C=1 S=1
      → 32 probes, mean 1.34  (234s)  [1/20 done, ETA 4450s]

  [  2/20] script 002 — chemistry of common elements
      turn  5/40  probes  T=1 E=1 C=1 S=1
      turn 10/40  probes  T=1 E=1 C=1 S=1
      turn 15/40  probes  T=1 E=1 C=1 S=1
      turn 20/40  probes  T=2 E=1 C=1 S=1
      turn 25/40  probes  T=1 E=1 C=1 S=1
      turn 30/40  probes  T=1 E=2 C=1 S=1
      turn 35/40  probes  T=1 E=2 C=1 S=1
  

### Judge reliability (§6.5) — weighted κ gate on a 5% re-score

In [ ]:
!python evaluate.py judge-reliability --scores-dir results/persona_R0 --frac 0.05

### QPM ablation (RQ6) — QPM-on vs `--no-qpm`

Same checkpoint, persona channel disabled. Compares expressed persona/affect with vs without the QPM conditioning (does compiling the QPM into the weights shape behaviour?).

In [ ]:
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --no-qpm --out-dir results/ablation_noqpm --resume
from evaluate import _persona_curve
on  = _persona_curve('results/persona_R0')['overall']
off = _persona_curve('results/ablation_noqpm/persona_R0')['overall']
print(f'RQ6 QPM ablation | QPM-on R0 overall = {on:.3f} | QPM-off R0 overall = {off:.3f} | delta = {on-off:+.3f}')

---
## P6 · Analyse — H1–H4 decision table (plan §3) + figures

`evaluate.py analyse` writes `results/analysis_data.json` **and** renders the Exp 1/2/5-style figures (png/pdf/svg) to `results/`: PersonaScore per-turn curve (R0 vs R1, T\* marked), T/E/C/S dimension bars, QA metrics vs thresholds, and the QPM-on vs `--no-qpm` ablation. The next cell displays them inline.

In [ ]:
!python evaluate.py analyse --results-dir results

import json
a = json.load(open('results/analysis_data.json'))
print('='*64)
for h in ('H1', 'H2', 'H3'):
    if h in a:
        print(f'{h}: {a[h]}')
if a.get('H4'):
    print('H4:', a['H4'])
print('-'*64)
print('DECISION:', json.dumps(a.get('decision', {}), indent=2))
print('='*64)

In [ ]:
# Display the Exp 6 figures (rendered by `evaluate.py analyse`, saved png/pdf/svg to results/).
from IPython.display import Image, display
figs = ['exp6_personascore_turn_series', 'exp6_dimension_bars',
        'exp6_qa_metrics', 'exp6_qpm_ablation']
for stem in figs:
    png = f'results/{stem}.png'
    if os.path.exists(png):
        print(stem)
        display(Image(filename=png))
    else:
        print(f'(missing {png} — run the P5 eval + P6 analyse cells first)')

---
### Next

Write `EXPERIMENT_REPORT.md` from `results/analysis_data.json` (after the run). Apply the §3 decision:

- **✓✓✓** — lock ADA knowledge-agent v0; apply the H4 refresh recommendation; proceed to the next scenario (therapy — QPM re-enters as full dynamics — or design the stack language).
- **✓✓✗** — competent QA, weak persona → add persona/episodic SFT data; retrain SFT only.
- **✓✗—** — over-confident → add unanswerable negatives + Sonnet refusal data; retrain SFT.
- **✗——** — 80M from-scratch below usability → trigger the RQ5 fallback (fine-tune SmolLM2-135M on the identical data).